# Module 3 — Agentic AI Systems · Coding Questions



### [Class 2] Question 1. The wire contract — serialize, revalidate on arrival, and what tampering actually catches

**Context:** An agent almost never keeps its answer in process. It serializes it, puts it on a queue
or an HTTP response, and something else picks it up — and the thing that picks it up has no reason to
trust what arrived. Pydantic owns both ends of that hop: the producer dumps with `model_dump_json()`,
the consumer re-validates with `model_validate_json()`. What is worth being precise about is *what
that catches*. Re-validation enforces the contract, not integrity — a value that was altered in
transit but still satisfies the schema arrives looking perfectly legitimate, and no amount of
re-validation will tell you otherwise.

**Problem Statement:**

Write a function `wire_roundtrip(answer, confidence, sources, tampered_confidence)` that sends one
agent response across a simulated boundary and reports what survived it.

1. Define one Pydantic (v2) model `AgentResponse` with **exactly** these fields, configured to reject
   any extra key (`extra="forbid"`):
   - `answer` — string
   - `confidence` — float constrained to the range 0.0 to 1.0 inclusive, declared as a field
     constraint rather than checked in a validator of your own
   - `sources` — list of strings, defaulting to an empty list
2. Build an `AgentResponse` from `answer`, `confidence` and `sources`, then serialize it to the JSON
   string that would go on the wire. Use Pydantic's own JSON serializer, not `json.dumps` over a dict —
   the exact string is part of the answer.
3. Hand that JSON string back to the model as a consumer would, validating the **JSON text** rather
   than a pre-parsed dict, and read the confidence off the object that comes back.
4. Build a tampered payload: take the response as a plain dictionary, replace its `confidence` with
   `tampered_confidence`, and serialize that to JSON. Validate it the same way. If it is accepted, the
   verdict is the string `"accepted"`; if it is rejected, the verdict is the `type` of the first error,
   passed through unchanged — whatever Pydantic put there, not a string of your own choosing.
5. Return a dictionary with exactly these four keys:
   - `"wire"` — the JSON string from step 2
   - `"recovered_confidence"` — the confidence read back in step 3
   - `"compact"` — the response as a dictionary with the `sources` field left out
   - `"tampered_verdict"` — the verdict from step 4

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: answer (string)
    Single line: a string
  Argument 2: confidence (float)
    Single line: one float
  Argument 3: sources (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 4: tampered_confidence (float)
    Single line: one float
```

**Output Format:**

For each test case, print the result of `wire_roundtrip()` on a new line.

**Sample Input:**

```
answer = "Paris"
confidence = 0.92
sources = ["wiki/Paris"]
tampered_confidence = 1.5
```

**Sample Output:**

```
{'wire': '{"answer":"Paris","confidence":0.92,"sources":["wiki/Paris"]}', 'recovered_confidence': 0.92, 'compact': {'answer': 'Paris', 'confidence': 0.92}, 'tampered_verdict': 'less_than_equal'}
```

**Note:** Build the model once at module level, outside the function. The wire string is compared
exactly, so it must be whatever Pydantic's serializer produces rather than a dict re-encoded by hand.
`"compact"` drops `sources` specifically — not "whatever happens to be empty".

---

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: wire_roundtrip
RETURN TYPE: Dictionary

ARGUMENTS:
1. answer              → String
2. confidence          → Float
3. sources             → List of Strings
4. tampered_confidence → Float
```

---

### 3. Reference Solution Code

```python
import json
from typing import Annotated

from pydantic import BaseModel, ConfigDict, Field, ValidationError


class AgentResponse(BaseModel):
    model_config = ConfigDict(extra="forbid")
    answer: str
    confidence: Annotated[float, Field(ge=0.0, le=1.0)]
    sources: list[str] = []


def wire_roundtrip(answer: str, confidence: float, sources: list,
                   tampered_confidence: float) -> dict:
    produced = AgentResponse(answer=answer, confidence=confidence, sources=sources)
    wire = produced.model_dump_json()
    received = AgentResponse.model_validate_json(wire)

    tampered = json.dumps({**produced.model_dump(), "confidence": tampered_confidence})
    try:
        AgentResponse.model_validate_json(tampered)
        verdict = "accepted"
    except ValidationError as exc:
        verdict = exc.errors()[0]["type"]

    return {"wire": wire,
            "recovered_confidence": received.confidence,
            "compact": produced.model_dump(exclude={"sources"}),
            "tampered_verdict": verdict}
```

---

### 4. Test Cases

**in1.txt**

```
1
Paris
0.92
1
wiki/Paris
1.5
```

**out1.txt**

```
{'wire': '{"answer":"Paris","confidence":0.92,"sources":["wiki/Paris"]}', 'recovered_confidence': 0.92, 'compact': {'answer': 'Paris', 'confidence': 0.92}, 'tampered_verdict': 'less_than_equal'}
```

**in2.txt**

```
1
Berlin
0.4
0
-0.2
```

**out2.txt**

```
{'wire': '{"answer":"Berlin","confidence":0.4,"sources":[]}', 'recovered_confidence': 0.4, 'compact': {'answer': 'Berlin', 'confidence': 0.4}, 'tampered_verdict': 'greater_than_equal'}
```

**in3.txt**

```
1
Tokyo
0.75
2
a
b
0.5
```

**out3.txt**

```
{'wire': '{"answer":"Tokyo","confidence":0.75,"sources":["a","b"]}', 'recovered_confidence': 0.75, 'compact': {'answer': 'Tokyo', 'confidence': 0.75}, 'tampered_verdict': 'accepted'}
```



In [26]:
import json
from typing import Annotated

from pydantic import BaseModel, ConfigDict, Field, ValidationError


class AgentResponse(BaseModel):
    model_config = ConfigDict(extra="forbid")
    answer: str
    confidence: Annotated[float, Field(ge=0.0, le=1.0)]
    sources: list[str] = []


def wire_roundtrip(answer: str, confidence: float, sources: list,
                   tampered_confidence: float) -> dict:
    produced = AgentResponse(answer=answer, confidence=confidence, sources=sources)
    wire = produced.model_dump_json()
    received = AgentResponse.model_validate_json(wire)

    tampered = json.dumps({**produced.model_dump(), "confidence": tampered_confidence})
    try:
        AgentResponse.model_validate_json(tampered)
        verdict = "accepted"
    except ValidationError as exc:
        verdict = exc.errors()[0]["type"]

    return {"wire": wire,
            "recovered_confidence": received.confidence,
            "compact": produced.model_dump(exclude={"sources"}),
            "tampered_verdict": verdict}


In [27]:
answer = 'Paris'
confidence = 0.92
sources = ['wiki/Paris']
tampered_confidence = 1.5

wire_roundtrip(answer, confidence, sources, tampered_confidence)


{'wire': '{"answer":"Paris","confidence":0.92,"sources":["wiki/Paris"]}',
 'recovered_confidence': 0.92,
 'compact': {'answer': 'Paris', 'confidence': 0.92},
 'tampered_verdict': 'less_than_equal'}

In [28]:
answer = 'Berlin'
confidence = 0.4
sources = []
tampered_confidence = -0.2

wire_roundtrip(answer, confidence, sources, tampered_confidence)


{'wire': '{"answer":"Berlin","confidence":0.4,"sources":[]}',
 'recovered_confidence': 0.4,
 'compact': {'answer': 'Berlin', 'confidence': 0.4},
 'tampered_verdict': 'greater_than_equal'}

In [29]:
answer = 'Tokyo'
confidence = 0.75
sources = ['a', 'b']
tampered_confidence = 0.5

wire_roundtrip(answer, confidence, sources, tampered_confidence)


{'wire': '{"answer":"Tokyo","confidence":0.75,"sources":["a","b"]}',
 'recovered_confidence': 0.75,
 'compact': {'answer': 'Tokyo', 'confidence': 0.75},
 'tampered_verdict': 'accepted'}

### [Class 9] Question 2. Which file answered the question

**Context:** A support assistant that answers from a folder of internal policy documents is only trusted
if it can say where each answer came from. A vector index makes that possible: every chunk keeps the
metadata of the document it was cut from, and a retrieved chunk carries both that metadata and the
similarity score that got it selected. The score is what lets an operator see the difference between a
confident match and a shrug.

**Problem Statement:**

Using LlamaIndex, write a function `retrieve_sources(file_names, file_texts, query, top_k)` that
retrieves the most relevant documents for a query and reports where they came from.

1. Build one `Document` per file, whose text is the file's text and whose metadata is exactly
   `{"file_name": <the file's name>}` — the metadata is embedded along with the text, so anything
   extra in it changes the scores.
2. Build a `VectorStoreIndex` over those documents.
3. Retrieve the `top_k` most similar nodes for `query`, using the index's own retriever
   (`as_retriever`), configured for `top_k` results.
4. Return a dictionary mapping each retrieved node's `"file_name"` to its similarity score, rounded to
   4 decimal places, with the entries in the order the retriever returned them.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: file_names (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 2: file_texts (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 3: query (string)
    Single line: a string
  Argument 4: top_k (integer)
    Single line: one integer
```

**Output Format:**

For each test case, print the result of `retrieve_sources()` on a new line.

**Sample Input:**

```
file_names = ["hr_policy.txt", "refund_policy.txt", "security_policy.txt", "product_roadmap.txt"]
file_texts = [
    "Employees are entitled to 24 days of paid leave annually. Maternity leave is 26 weeks. Remote work is permitted up to 3 days per week with manager approval.",
    "Customers may request a full refund within 30 days of purchase. Subscription cancellations after 30 days are prorated. Refunds are processed within 7 business days.",
    "All customer data is encrypted at rest using AES-256. Data in transit uses TLS 1.3. Production database access is restricted to the SRE team.",
    "Q1: launch multi-region failover for the storage product. Q2: release fine-grained IAM policies and audit log streaming."
]
query = "Can I get my money back if I cancel early?"
top_k = 2
```

**Sample Output:**

```
{'refund_policy.txt': 0.7553, 'hr_policy.txt': 0.4989}
```

**Note:** No LLM is configured in this environment, and none is needed — retrieval runs on embeddings
alone. Set the embedding model once at module level, outside the function, so the model is not reloaded
on every call:

```python
Settings.embed_model = FastEmbedEmbedding(model_name="BAAI/bge-small-en-v1.5")
```

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: retrieve_sources
RETURN TYPE: Dictionary

ARGUMENTS:
1. file_names → List of Strings
2. file_texts → List of Strings
3. query      → String
4. top_k      → Integer
```

---

### 3. Reference Solution Code

```python
from llama_index.core import Document, Settings, VectorStoreIndex
from llama_index.embeddings.fastembed import FastEmbedEmbedding

Settings.embed_model = FastEmbedEmbedding(model_name="BAAI/bge-small-en-v1.5")


def retrieve_sources(file_names: list, file_texts: list, query: str, top_k: int) -> dict:
    documents = [
        Document(text=text, metadata={"file_name": name})
        for name, text in zip(file_names, file_texts)
    ]
    index = VectorStoreIndex.from_documents(documents)
    nodes = index.as_retriever(similarity_top_k=top_k).retrieve(query)
    return {node.metadata["file_name"]: round(node.score, 4) for node in nodes}
```

---

### 4. Test Cases

**in1.txt**

```
1
4
hr_policy.txt
refund_policy.txt
security_policy.txt
product_roadmap.txt
4
Employees are entitled to 24 days of paid leave annually. Maternity leave is 26 weeks. Remote work is permitted up to 3 days per week with manager approval.
Customers may request a full refund within 30 days of purchase. Subscription cancellations after 30 days are prorated. Refunds are processed within 7 business days.
All customer data is encrypted at rest using AES-256. Data in transit uses TLS 1.3. Production database access is restricted to the SRE team.
Q1: launch multi-region failover for the storage product. Q2: release fine-grained IAM policies and audit log streaming.
Can I get my money back if I cancel early?
2
```

**out1.txt**

```
{'refund_policy.txt': 0.7553, 'hr_policy.txt': 0.4989}
```

**in2.txt**

```
1
4
hr_policy.txt
refund_policy.txt
security_policy.txt
product_roadmap.txt
4
Employees are entitled to 24 days of paid leave annually. Maternity leave is 26 weeks. Remote work is permitted up to 3 days per week with manager approval.
Customers may request a full refund within 30 days of purchase. Subscription cancellations after 30 days are prorated. Refunds are processed within 7 business days.
All customer data is encrypted at rest using AES-256. Data in transit uses TLS 1.3. Production database access is restricted to the SRE team.
Q1: launch multi-region failover for the storage product. Q2: release fine-grained IAM policies and audit log streaming.
who can touch the production database
3
```

**out2.txt**

```
{'security_policy.txt': 0.6526, 'product_roadmap.txt': 0.6146, 'hr_policy.txt': 0.5297}
```

In [1]:
from llama_index.core import Document, Settings, VectorStoreIndex
from llama_index.embeddings.fastembed import FastEmbedEmbedding

Settings.embed_model = FastEmbedEmbedding(model_name="BAAI/bge-small-en-v1.5")


def retrieve_sources(file_names: list, file_texts: list, query: str, top_k: int) -> dict:
    documents = [
        Document(text=text, metadata={"file_name": name})
        for name, text in zip(file_names, file_texts)
    ]
    index = VectorStoreIndex.from_documents(documents)
    nodes = index.as_retriever(similarity_top_k=top_k).retrieve(query)
    return {node.metadata["file_name"]: round(node.score, 4) for node in nodes}

In [2]:
file_names = ["hr_policy.txt", "refund_policy.txt", "security_policy.txt", "product_roadmap.txt"]
file_texts = [
    "Employees are entitled to 24 days of paid leave annually. Maternity leave is 26 weeks. Remote work is permitted up to 3 days per week with manager approval.",
    "Customers may request a full refund within 30 days of purchase. Subscription cancellations after 30 days are prorated. Refunds are processed within 7 business days.",
    "All customer data is encrypted at rest using AES-256. Data in transit uses TLS 1.3. Production database access is restricted to the SRE team.",
    "Q1: launch multi-region failover for the storage product. Q2: release fine-grained IAM policies and audit log streaming.",
]
query = "Can I get my money back if I cancel early?"
top_k = 2

retrieve_sources(file_names, file_texts, query, top_k)

{'refund_policy.txt': 0.7553, 'hr_policy.txt': 0.4989}

In [3]:
file_names = ["hr_policy.txt", "refund_policy.txt", "security_policy.txt", "product_roadmap.txt"]
file_texts = [
    "Employees are entitled to 24 days of paid leave annually. Maternity leave is 26 weeks. Remote work is permitted up to 3 days per week with manager approval.",
    "Customers may request a full refund within 30 days of purchase. Subscription cancellations after 30 days are prorated. Refunds are processed within 7 business days.",
    "All customer data is encrypted at rest using AES-256. Data in transit uses TLS 1.3. Production database access is restricted to the SRE team.",
    "Q1: launch multi-region failover for the storage product. Q2: release fine-grained IAM policies and audit log streaming.",
]
query = "who can touch the production database"
top_k = 3

retrieve_sources(file_names, file_texts, query, top_k)

{'security_policy.txt': 0.6526,
 'product_roadmap.txt': 0.6146,
 'hr_policy.txt': 0.5297}

### [Class 9] Question 3. Two indexes over one corpus, and what each one reaches

**Context:** A vector index and a keyword table index answer the same question in incompatible ways. The
vector index always hands back exactly the number of chunks you asked for, ranked by meaning — even when
the corpus has nothing to say. The keyword table index only returns chunks that share an extracted
keyword with the query, so it can hand back nothing at all, or nearly everything when the shared word is
a common one. Choosing between them starts with seeing what each actually reaches.

**Problem Statement:**

Using LlamaIndex, write a function `retriever_reach(doc_texts, queries, top_k)` that runs each query
through both index types and reports which documents each one reached.

1. Build one `Document` per entry in `doc_texts`, keeping their original order.
2. Build two indexes over that same list of documents: a `VectorStoreIndex` and a
   `SimpleKeywordTableIndex`.
3. Get a retriever from each index with `as_retriever`: the vector retriever configured for `top_k`
   results, the keyword retriever with its defaults.
4. For each query, retrieve from both and map each returned node back to the 0-based position of the
   document it came from in `doc_texts`: a retrieved node's `ref_doc_id` is the `doc_id` of the
   `Document` it was cut from.
5. Return a dictionary keyed by the 0-based position of the query, whose value is a dictionary with
   exactly two keys: `"vector"`, the positions the vector retriever returned in ranked order, and
   `"keyword"`, the positions the keyword retriever returned, sorted ascending.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: doc_texts (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 2: queries (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 3: top_k (integer)
    Single line: one integer
```

**Output Format:**

For each test case, print the result of `retriever_reach()` on a new line.

**Sample Input:**

```
doc_texts = [
    "Incident INC-2847 was a prompt injection attack against the chat endpoint. Root cause: an output filter regression after the v2.3 deploy.",
    "Incident INC-3102 was a Kubernetes autoscaler misconfiguration that caused cascading pod evictions during peak traffic.",
    "Incident INC-2915 was a data pipeline stall after a schema evolution in the orders table. Downstream jobs failed silently for six hours.",
    "Incident INC-3300 was a certificate expiry that broke mutual TLS between the mesh sidecars for eleven minutes.",
    "Incident INC-2650 was a cache stampede after a deploy invalidated every key at once, tripling database load."
]
queries = ["INC-2915", "what happened to the certificates", "how do I add a second user to my account"]
top_k = 2
```

**Sample Output:**

```
LLM is explicitly disabled. Using MockLLM.
{0: {'vector': [2, 0], 'keyword': [0, 1, 2, 3, 4]}, 1: {'vector': [3, 2], 'keyword': []}, 2: {'vector': [0, 1], 'keyword': []}}
```

**Note:** No LLM is available in this environment, so disable it at module level before building the keyword
index — `Settings.llm = None` — otherwise LlamaIndex tries to resolve a hosted model and raises. That
assignment makes LlamaIndex print one line to stdout; leave it where it lands, as the sample shows.
Load the embedding model once at module level, outside the function:

```python
Settings.embed_model = FastEmbedEmbedding(model_name="BAAI/bge-small-en-v1.5")
```

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: retriever_reach
RETURN TYPE: Dictionary

ARGUMENTS:
1. doc_texts → List of Strings
2. queries   → List of Strings
3. top_k     → Integer
```

---

### 3. Reference Solution Code

```python
from llama_index.core import Document, Settings, SimpleKeywordTableIndex, VectorStoreIndex
from llama_index.embeddings.fastembed import FastEmbedEmbedding

Settings.llm = None
Settings.embed_model = FastEmbedEmbedding(model_name="BAAI/bge-small-en-v1.5")


def retriever_reach(doc_texts: list, queries: list, top_k: int) -> dict:
    documents = [Document(text=text) for text in doc_texts]
    position = {document.doc_id: i for i, document in enumerate(documents)}

    vector_retriever = VectorStoreIndex.from_documents(documents).as_retriever(similarity_top_k=top_k)
    keyword_retriever = SimpleKeywordTableIndex.from_documents(documents).as_retriever()

    reach = {}
    for i, query in enumerate(queries):
        reach[i] = {
            "vector": [position[n.node.ref_doc_id] for n in vector_retriever.retrieve(query)],
            "keyword": sorted(position[n.node.ref_doc_id] for n in keyword_retriever.retrieve(query)),
        }
    return reach
```

---

### 4. Test Cases

**in1.txt**

```
1
5
Incident INC-2847 was a prompt injection attack against the chat endpoint. Root cause: an output filter regression after the v2.3 deploy.
Incident INC-3102 was a Kubernetes autoscaler misconfiguration that caused cascading pod evictions during peak traffic.
Incident INC-2915 was a data pipeline stall after a schema evolution in the orders table. Downstream jobs failed silently for six hours.
Incident INC-3300 was a certificate expiry that broke mutual TLS between the mesh sidecars for eleven minutes.
Incident INC-2650 was a cache stampede after a deploy invalidated every key at once, tripling database load.
3
INC-2915
what happened to the certificates
how do I add a second user to my account
2
```

**out1.txt**

```
LLM is explicitly disabled. Using MockLLM.
{0: {'vector': [2, 0], 'keyword': [0, 1, 2, 3, 4]}, 1: {'vector': [3, 2], 'keyword': []}, 2: {'vector': [0, 1], 'keyword': []}}
```

**in2.txt**

```
1
4
Engineers on the primary on-call rotation hand off at 09:00 local time, and the schedule is published a month in advance.
The weekend coverage roster rotates every two weeks and is agreed at the Monday planning meeting.
Encryption keys are re-issued on a fixed cadence: AES-256 material is replaced every 90 days by the platform team.
Refunds are processed within 7 business days to the original payment method.
2
AES-256 rotation schedule
cryptocurrency payouts to vendors
3
```

**out2.txt**

```
LLM is explicitly disabled. Using MockLLM.
{0: {'vector': [2, 0, 1], 'keyword': [0, 2]}, 1: {'vector': [2, 3, 1], 'keyword': []}}
```

In [4]:
from llama_index.core import Document, Settings, SimpleKeywordTableIndex, VectorStoreIndex
from llama_index.embeddings.fastembed import FastEmbedEmbedding

Settings.llm = None
Settings.embed_model = FastEmbedEmbedding(model_name="BAAI/bge-small-en-v1.5")


def retriever_reach(doc_texts: list, queries: list, top_k: int) -> dict:
    documents = [Document(text=text) for text in doc_texts]
    position = {document.doc_id: i for i, document in enumerate(documents)}

    vector_retriever = VectorStoreIndex.from_documents(documents).as_retriever(similarity_top_k=top_k)
    keyword_retriever = SimpleKeywordTableIndex.from_documents(documents).as_retriever()

    reach = {}
    for i, query in enumerate(queries):
        reach[i] = {
            "vector": [position[n.node.ref_doc_id] for n in vector_retriever.retrieve(query)],
            "keyword": sorted(position[n.node.ref_doc_id] for n in keyword_retriever.retrieve(query)),
        }
    return reach

LLM is explicitly disabled. Using MockLLM.


In [5]:
doc_texts = [
    "Incident INC-2847 was a prompt injection attack against the chat endpoint. Root cause: an output filter regression after the v2.3 deploy.",
    "Incident INC-3102 was a Kubernetes autoscaler misconfiguration that caused cascading pod evictions during peak traffic.",
    "Incident INC-2915 was a data pipeline stall after a schema evolution in the orders table. Downstream jobs failed silently for six hours.",
    "Incident INC-3300 was a certificate expiry that broke mutual TLS between the mesh sidecars for eleven minutes.",
    "Incident INC-2650 was a cache stampede after a deploy invalidated every key at once, tripling database load.",
]
queries = ["INC-2915", "what happened to the certificates", "how do I add a second user to my account"]
top_k = 2

retriever_reach(doc_texts, queries, top_k)

{0: {'vector': [2, 0], 'keyword': [0, 1, 2, 3, 4]},
 1: {'vector': [3, 2], 'keyword': []},
 2: {'vector': [0, 1], 'keyword': []}}

In [6]:
doc_texts = [
    "Engineers on the primary on-call rotation hand off at 09:00 local time, and the schedule is published a month in advance.",
    "The weekend coverage roster rotates every two weeks and is agreed at the Monday planning meeting.",
    "Encryption keys are re-issued on a fixed cadence: AES-256 material is replaced every 90 days by the platform team.",
    "Refunds are processed within 7 business days to the original payment method.",
]
queries = ["AES-256 rotation schedule", "cryptocurrency payouts to vendors"]
top_k = 3

retriever_reach(doc_texts, queries, top_k)

{0: {'vector': [2, 0, 1], 'keyword': [0, 2]},
 1: {'vector': [2, 3, 1], 'keyword': []}}

### [Class 9] Question 4. Chunk size decides what the retriever can hand back

**Context:** The chunk size chosen at index time is not a storage detail — it is the unit of evidence the
answer will be built from. Cut a policy document too coarsely and a single chunk covers five unrelated
sections, so the retrieved evidence is mostly noise and the similarity score is dragged down by
everything the query did not ask about. Cut it finely and the retriever can hand back the one paragraph
that answers the question.

**Problem Statement:**

Using LlamaIndex, write a function `chunk_profile(document_text, chunk_size, chunk_overlap, query)` that
indexes one document at a given chunk size and reports what the retriever gets back.

1. Set the global chunking settings from the arguments: `Settings.chunk_size` and
   `Settings.chunk_overlap`.
2. Build a `VectorStoreIndex` from a single `Document` holding `document_text`, so the document is split
   into nodes according to those settings.
3. Retrieve the single most similar node for `query`, using the index's own retriever
   (`as_retriever`).
4. Return a dictionary with exactly these four keys:
   - `"n_nodes"` — how many nodes the document was split into; the index keeps them in its
     `docstore`
   - `"top_score"` — the retrieved node's similarity score, rounded to 4 decimal places
   - `"top_chars"` — the number of characters in the retrieved node's text
   - `"top_starts_with"` — the first 40 characters of the retrieved node's text

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: document_text (string)
    Single line: a string
  Argument 2: chunk_size (integer)
    Single line: one integer
  Argument 3: chunk_overlap (integer)
    Single line: one integer
  Argument 4: query (string)
    Single line: a string
```

**Output Format:**

For each test case, print the result of `chunk_profile()` on a new line.

**Sample Input:**

```
document_text = "Section 1. All customer data is encrypted at rest using AES-256 and in transit with TLS 1.3. Key material is rotated every 90 days by the platform team. Section 2. Production database access is restricted to the on-call SRE group. Every access grant expires after eight hours and is logged to the audit stream. Section 3. Security incidents must be reported to the CISO within one hour of detection. The incident commander opens a channel, records a timeline, and files a postmortem within five working days. Section 4. Employees must use multi-factor authentication for all internal systems. Hardware tokens are issued to anyone with production access. Section 5. Vendor software is reviewed before purchase and re-reviewed once a year."
chunk_size = 64
chunk_overlap = 16
query = "how long does a production access grant last"
```

**Sample Output:**

```
{'n_nodes': 4, 'top_score': 0.812, 'top_chars': 235, 'top_starts_with': 'Production database access is restricted'}
```

**Note:** Chunk sizes are counted in tokens, not characters, which is why `"top_chars"` and
`chunk_size` are not the same number. Load the embedding model once at module level, outside the
function:

```python
Settings.embed_model = FastEmbedEmbedding(model_name="BAAI/bge-small-en-v1.5")
```

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: chunk_profile
RETURN TYPE: Dictionary

ARGUMENTS:
1. document_text → String
2. chunk_size    → Integer
3. chunk_overlap → Integer
4. query         → String
```

---

### 3. Reference Solution Code

```python
from llama_index.core import Document, Settings, VectorStoreIndex
from llama_index.embeddings.fastembed import FastEmbedEmbedding

Settings.embed_model = FastEmbedEmbedding(model_name="BAAI/bge-small-en-v1.5")


def chunk_profile(document_text: str, chunk_size: int, chunk_overlap: int, query: str) -> dict:
    Settings.chunk_size = chunk_size
    Settings.chunk_overlap = chunk_overlap

    index = VectorStoreIndex.from_documents([Document(text=document_text)])
    top = index.as_retriever(similarity_top_k=1).retrieve(query)[0]

    return {
        "n_nodes": len(index.docstore.docs),
        "top_score": round(top.score, 4),
        "top_chars": len(top.node.text),
        "top_starts_with": top.node.text[:40],
    }
```

---

### 4. Test Cases

**in1.txt**

```
1
Section 1. All customer data is encrypted at rest using AES-256 and in transit with TLS 1.3. Key material is rotated every 90 days by the platform team. Section 2. Production database access is restricted to the on-call SRE group. Every access grant expires after eight hours and is logged to the audit stream. Section 3. Security incidents must be reported to the CISO within one hour of detection. The incident commander opens a channel, records a timeline, and files a postmortem within five working days. Section 4. Employees must use multi-factor authentication for all internal systems. Hardware tokens are issued to anyone with production access. Section 5. Vendor software is reviewed before purchase and re-reviewed once a year.
64
16
how long does a production access grant last
```

**out1.txt**

```
{'n_nodes': 4, 'top_score': 0.812, 'top_chars': 235, 'top_starts_with': 'Production database access is restricted'}
```

**in2.txt**

```
1
Section 1. All customer data is encrypted at rest using AES-256 and in transit with TLS 1.3. Key material is rotated every 90 days by the platform team. Section 2. Production database access is restricted to the on-call SRE group. Every access grant expires after eight hours and is logged to the audit stream. Section 3. Security incidents must be reported to the CISO within one hour of detection. The incident commander opens a channel, records a timeline, and files a postmortem within five working days. Section 4. Employees must use multi-factor authentication for all internal systems. Hardware tokens are issued to anyone with production access. Section 5. Vendor software is reviewed before purchase and re-reviewed once a year.
256
32
how long does a production access grant last
```

**out2.txt**

```
{'n_nodes': 1, 'top_score': 0.7325, 'top_chars': 737, 'top_starts_with': 'Section 1. All customer data is encrypte'}
```

In [7]:
from llama_index.core import Document, Settings, VectorStoreIndex
from llama_index.embeddings.fastembed import FastEmbedEmbedding

Settings.embed_model = FastEmbedEmbedding(model_name="BAAI/bge-small-en-v1.5")


def chunk_profile(document_text: str, chunk_size: int, chunk_overlap: int, query: str) -> dict:
    Settings.chunk_size = chunk_size
    Settings.chunk_overlap = chunk_overlap

    index = VectorStoreIndex.from_documents([Document(text=document_text)])
    top = index.as_retriever(similarity_top_k=1).retrieve(query)[0]

    return {
        "n_nodes": len(index.docstore.docs),
        "top_score": round(top.score, 4),
        "top_chars": len(top.node.text),
        "top_starts_with": top.node.text[:40],
    }

In [8]:
document_text = (
    "Section 1. All customer data is encrypted at rest using AES-256 and in transit with TLS 1.3. "
    "Key material is rotated every 90 days by the platform team. "
    "Section 2. Production database access is restricted to the on-call SRE group. "
    "Every access grant expires after eight hours and is logged to the audit stream. "
    "Section 3. Security incidents must be reported to the CISO within one hour of detection. "
    "The incident commander opens a channel, records a timeline, and files a postmortem within five working days. "
    "Section 4. Employees must use multi-factor authentication for all internal systems. "
    "Hardware tokens are issued to anyone with production access. "
    "Section 5. Vendor software is reviewed before purchase and re-reviewed once a year."
)
chunk_size = 64
chunk_overlap = 16
query = "how long does a production access grant last"

chunk_profile(document_text, chunk_size, chunk_overlap, query)

{'n_nodes': 4,
 'top_score': 0.812,
 'top_chars': 235,
 'top_starts_with': 'Production database access is restricted'}

In [9]:
document_text = (
    "Section 1. All customer data is encrypted at rest using AES-256 and in transit with TLS 1.3. "
    "Key material is rotated every 90 days by the platform team. "
    "Section 2. Production database access is restricted to the on-call SRE group. "
    "Every access grant expires after eight hours and is logged to the audit stream. "
    "Section 3. Security incidents must be reported to the CISO within one hour of detection. "
    "The incident commander opens a channel, records a timeline, and files a postmortem within five working days. "
    "Section 4. Employees must use multi-factor authentication for all internal systems. "
    "Hardware tokens are issued to anyone with production access. "
    "Section 5. Vendor software is reviewed before purchase and re-reviewed once a year."
)
chunk_size = 256
chunk_overlap = 32
query = "how long does a production access grant last"

chunk_profile(document_text, chunk_size, chunk_overlap, query)

{'n_nodes': 1,
 'top_score': 0.7325,
 'top_chars': 737,
 'top_starts_with': 'Section 1. All customer data is encrypte'}

### [Class 9] Question 5. Walking out from an entity in a knowledge graph

**Context:** Once an extraction step has turned prose into typed triples, the graph store becomes the
retrieval surface: a question about an entity is answered by walking out from that entity and serializing
what is found. How far the walk goes is a knob — one hop returns the facts stated directly about the
entity, two hops also picks up the facts stated about whatever those first facts pointed at. The
direction matters too: an edge stored as `A -> B` is reachable from `A`, not from `B`.

**Problem Statement:**

Using LlamaIndex's `SimpleGraphStore`, write a function `entity_context(triplets, seed, depth)` that
loads a set of extracted triples and serializes the neighbourhood around one entity.

1. Create a `SimpleGraphStore` and insert every triplet into it with
   `upsert_triplet(subject, relation, object)`. Each record in `triplets` has the keys
   `"source"`, `"type"` and `"target"`, in that order as subject, relation and object.
2. Ask the store for the relation map of the single entity `seed`, walking out to `depth` hops:
   `get_rel_map(subjs=[seed], depth=depth)`.
3. Return a dictionary with exactly two keys:
   - `"triples"` — the hops the store returned for `seed`, each formatted as
     `(subject) -[:RELATION]-> (object)` from that hop's **own** subject, relation and object —
     every hop carries its own subject, which is not always `seed` —
     in the order the store returned them
   - `"entities"` — every distinct subject and object name appearing in those hops, sorted ascending

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: triplets (list_dict)
    Line 1: N (number of triplets)
    Next N lines: one JSON object per line, each with a "source", "type" and "target" key
  Argument 2: seed (string)
    Single line: a string
  Argument 3: depth (integer)
    Single line: one integer
```

**Output Format:**

For each test case, print the result of `entity_context()` on a new line.

**Sample Input:**

```
triplets = [
    {"source": "Tesla", "type": "HEADQUARTERED_IN", "target": "Austin"},
    {"source": "Austin", "type": "LOCATED_IN", "target": "Texas"},
    {"source": "Texas", "type": "PART_OF", "target": "United States"},
    {"source": "Tesla", "type": "PRODUCES", "target": "Model 3"},
    {"source": "Elon Musk", "type": "JOINED", "target": "Tesla"}
]
seed = "Tesla"
depth = 2
```

**Sample Output:**

```
{'triples': ['(Tesla) -[:HEADQUARTERED_IN]-> (Austin)', '(Austin) -[:LOCATED_IN]-> (Texas)', '(Tesla) -[:PRODUCES]-> (Model 3)'], 'entities': ['Austin', 'Model 3', 'Tesla', 'Texas']}
```

**Note:** The relation map is keyed by the entity you asked about. Read the hops off that key rather than
flattening the whole map, and let the store decide which hops are in range — the walk is its job, not
yours.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: entity_context
RETURN TYPE: Dictionary

ARGUMENTS:
1. triplets → List of Dicts
2. seed     → String
3. depth    → Integer
```

---

### 3. Reference Solution Code

```python
from llama_index.core.graph_stores import SimpleGraphStore


def entity_context(triplets: list, seed: str, depth: int) -> dict:
    graph_store = SimpleGraphStore()
    for triplet in triplets:
        graph_store.upsert_triplet(triplet["source"], triplet["type"], triplet["target"])

    paths = graph_store.get_rel_map(subjs=[seed], depth=depth).get(seed, [])
    return {
        "triples": ["({}) -[:{}]-> ({})".format(*path) for path in paths],
        "entities": sorted({name for path in paths for name in (path[0], path[2])}),
    }
```

---

### 4. Test Cases

**in1.txt**

```
1
5
{"source": "Tesla", "type": "HEADQUARTERED_IN", "target": "Austin"}
{"source": "Austin", "type": "LOCATED_IN", "target": "Texas"}
{"source": "Texas", "type": "PART_OF", "target": "United States"}
{"source": "Tesla", "type": "PRODUCES", "target": "Model 3"}
{"source": "Elon Musk", "type": "JOINED", "target": "Tesla"}
Tesla
2
```

**out1.txt**

```
{'triples': ['(Tesla) -[:HEADQUARTERED_IN]-> (Austin)', '(Austin) -[:LOCATED_IN]-> (Texas)', '(Tesla) -[:PRODUCES]-> (Model 3)'], 'entities': ['Austin', 'Model 3', 'Tesla', 'Texas']}
```

**in2.txt**

```
1
5
{"source": "Tesla", "type": "HEADQUARTERED_IN", "target": "Austin"}
{"source": "Austin", "type": "LOCATED_IN", "target": "Texas"}
{"source": "Texas", "type": "PART_OF", "target": "United States"}
{"source": "Tesla", "type": "PRODUCES", "target": "Model 3"}
{"source": "Elon Musk", "type": "JOINED", "target": "Tesla"}
Elon Musk
3
```

**out2.txt**

```
{'triples': ['(Elon Musk) -[:JOINED]-> (Tesla)', '(Tesla) -[:HEADQUARTERED_IN]-> (Austin)', '(Austin) -[:LOCATED_IN]-> (Texas)', '(Tesla) -[:PRODUCES]-> (Model 3)'], 'entities': ['Austin', 'Elon Musk', 'Model 3', 'Tesla', 'Texas']}
```

In [10]:
from llama_index.core.graph_stores import SimpleGraphStore


def entity_context(triplets: list, seed: str, depth: int) -> dict:
    graph_store = SimpleGraphStore()
    for triplet in triplets:
        graph_store.upsert_triplet(triplet["source"], triplet["type"], triplet["target"])

    paths = graph_store.get_rel_map(subjs=[seed], depth=depth).get(seed, [])
    return {
        "triples": ["({}) -[:{}]-> ({})".format(*path) for path in paths],
        "entities": sorted({name for path in paths for name in (path[0], path[2])}),
    }

In [11]:
triplets = [
    {"source": "Tesla", "type": "HEADQUARTERED_IN", "target": "Austin"},
    {"source": "Austin", "type": "LOCATED_IN", "target": "Texas"},
    {"source": "Texas", "type": "PART_OF", "target": "United States"},
    {"source": "Tesla", "type": "PRODUCES", "target": "Model 3"},
    {"source": "Elon Musk", "type": "JOINED", "target": "Tesla"},
]
seed = "Tesla"
depth = 2

entity_context(triplets, seed, depth)

{'triples': ['(Tesla) -[:HEADQUARTERED_IN]-> (Austin)',
  '(Austin) -[:LOCATED_IN]-> (Texas)',
  '(Tesla) -[:PRODUCES]-> (Model 3)'],
 'entities': ['Austin', 'Model 3', 'Tesla', 'Texas']}

In [12]:
triplets = [
    {"source": "Tesla", "type": "HEADQUARTERED_IN", "target": "Austin"},
    {"source": "Austin", "type": "LOCATED_IN", "target": "Texas"},
    {"source": "Texas", "type": "PART_OF", "target": "United States"},
    {"source": "Tesla", "type": "PRODUCES", "target": "Model 3"},
    {"source": "Elon Musk", "type": "JOINED", "target": "Tesla"},
]
seed = "Elon Musk"
depth = 3

entity_context(triplets, seed, depth)

{'triples': ['(Elon Musk) -[:JOINED]-> (Tesla)',
  '(Tesla) -[:HEADQUARTERED_IN]-> (Austin)',
  '(Austin) -[:LOCATED_IN]-> (Texas)',
  '(Tesla) -[:PRODUCES]-> (Model 3)'],
 'entities': ['Austin', 'Elon Musk', 'Model 3', 'Tesla', 'Texas']}

### [Class 10] Question 6. The question a vector index cannot answer

**Context:** "Which engineering managers have team members working on projects involving LLM security?"
looks like a retrieval question and is not one. No single passage contains the answer: the topic evidence
lives in project documents and in the skills recorded against people, the staffing lives in assignments,
and the reporting line lives in the org chart. Answering it means traversing a graph that joins all
three, then filtering the result.

**Problem Statement:**

Using NetworkX, write a function `managers_over_topic(employees, projects, assignments, documents, topic)`
that builds an organizational property graph and answers that question for any topic.

1. Build a `MultiDiGraph` whose nodes are `("employee", id)`, `("project", id)`, `("skill", name)` and
   `("document", id)` tuples, and whose edges carry the relationship name on an attribute called `rel`:
   - `("employee", id)` → `("employee", manager_id)` with `rel="REPORTS_TO"`, unless the employee's
     `manager_id` is null
   - `("employee", id)` → `("skill", name)` with `rel="HAS_SKILL"`, one per entry in the employee's
     `skills` list
   - `("employee", employee_id)` → `("project", project_id)` with `rel="WORKS_ON"`, one per assignment
   - `("document", id)` → `("project", project_id)` with `rel="ABOUT"`, one per document
   Keep each employee's `name` and `is_engineering` flag, each project's and each skill's `name`,
   and each document's `text`, as node attributes.
2. Find the projects the topic touches. A project qualifies if **either** a document `ABOUT` it contains
   `topic` in its text, compared case-insensitively, **or** somebody who `WORKS_ON` it has a
   `HAS_SKILL` edge to a skill whose name equals `topic` exactly.
3. Collect every employee with a `WORKS_ON` edge into one of those projects, then take each one's
   direct manager — a single `REPORTS_TO` hop, not the whole reporting chain.
4. Return the names of those managers whose `is_engineering` flag is set, sorted ascending, with no
   duplicates.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: employees (list_dict)
    Line 1: N (number of employees)
    Next N lines: one JSON object per line, each with an "id", "name", "manager_id",
                  "is_engineering" and "skills" key
  Argument 2: projects (list_dict)
    Line 1: N (number of projects)
    Next N lines: one JSON object per line, each with an "id" and "name" key
  Argument 3: assignments (list_dict)
    Line 1: N (number of assignments)
    Next N lines: one JSON object per line, each with an "employee_id" and "project_id" key
  Argument 4: documents (list_dict)
    Line 1: N (number of documents)
    Next N lines: one JSON object per line, each with an "id", "project_id" and "text" key
  Argument 5: topic (string)
    Single line: a string

Every non-null `manager_id` names an employee in `employees`, and every `project_id` names a
project in `projects`.
```

**Output Format:**

For each test case, print the result of `managers_over_topic()` on a new line.

**Sample Input:**

```
employees = [
    {"id": 1, "name": "Nadia Osei", "manager_id": None, "is_engineering": 0, "skills": []},
    {"id": 2, "name": "Priya Raman", "manager_id": 1, "is_engineering": 1, "skills": []},
    {"id": 3, "name": "Marco Silva", "manager_id": 2, "is_engineering": 1, "skills": []},
    {"id": 4, "name": "Elena Duarte", "manager_id": 3, "is_engineering": 1, "skills": ["Leadership", "Python"]},
    {"id": 5, "name": "Sofia Klein", "manager_id": 2, "is_engineering": 0, "skills": []},
    {"id": 7, "name": "Ana Iyer", "manager_id": 4, "is_engineering": 1, "skills": ["Python", "LLM Security"]},
    {"id": 8, "name": "Ben Cho", "manager_id": 4, "is_engineering": 1, "skills": ["Rust"]},
    {"id": 9, "name": "Chika Mba", "manager_id": 5, "is_engineering": 0, "skills": ["LLM Security"]},
    {"id": 12, "name": "Grace Lin", "manager_id": 13, "is_engineering": 0, "skills": []},
    {"id": 13, "name": "Hiro Tanaka", "manager_id": 2, "is_engineering": 0, "skills": []}
]
projects = [
    {"id": 101, "name": "Guardrail"},
    {"id": 102, "name": "Atlas"},
    {"id": 103, "name": "Prism"},
    {"id": 104, "name": "Sentinel"}
]
assignments = [
    {"employee_id": 7, "project_id": 101},
    {"employee_id": 9, "project_id": 101},
    {"employee_id": 8, "project_id": 104},
    {"employee_id": 12, "project_id": 103},
    {"employee_id": 7, "project_id": 102}
]
documents = [
    {"id": 201, "project_id": 101, "text": "Project Guardrail hardens the company LLM endpoints against prompt injection. This is core LLM Security work."},
    {"id": 202, "project_id": 104, "text": "Sentinel provides continuous LLM Security monitoring of production model traffic."},
    {"id": 203, "project_id": 102, "text": "Atlas is the unified data platform providing feature storage and batch pipelines."},
    {"id": 205, "project_id": 103, "text": "Prism is the shared design system defining components and typography."}
]
topic = "LLM Security"
```

**Sample Output:**

```
['Elena Duarte']
```

**Note:** The `is_engineering` flag is applied to the manager, not to the report.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: managers_over_topic
RETURN TYPE: List of Strings

ARGUMENTS:
1. employees   → List of Dicts
2. projects    → List of Dicts
3. assignments → List of Dicts
4. documents   → List of Dicts
5. topic       → String
```

---

### 3. Reference Solution Code

```python
import networkx as nx


def managers_over_topic(employees: list, projects: list, assignments: list,
                        documents: list, topic: str) -> list:
    graph = nx.MultiDiGraph()

    for employee in employees:
        graph.add_node(("employee", employee["id"]), name=employee["name"],
                       is_engineering=employee["is_engineering"])
        if employee["manager_id"] is not None:
            graph.add_edge(("employee", employee["id"]), ("employee", employee["manager_id"]),
                           rel="REPORTS_TO")
        for skill in employee["skills"]:
            graph.add_node(("skill", skill), name=skill)
            graph.add_edge(("employee", employee["id"]), ("skill", skill), rel="HAS_SKILL")

    for project in projects:
        graph.add_node(("project", project["id"]), name=project["name"])

    for assignment in assignments:
        graph.add_edge(("employee", assignment["employee_id"]),
                       ("project", assignment["project_id"]), rel="WORKS_ON")

    for document in documents:
        graph.add_node(("document", document["id"]), text=document["text"])
        graph.add_edge(("document", document["id"]), ("project", document["project_id"]),
                       rel="ABOUT")

    def out_neighbours(node, rel):
        return [t for _, t, d in graph.out_edges(node, data=True) if d.get("rel") == rel]

    target_projects = set()
    for node, data in graph.nodes(data=True):
        if node[0] == "document" and topic.lower() in data["text"].lower():
            target_projects.update(out_neighbours(node, "ABOUT"))
    for source, target, data in graph.edges(data=True):
        if data.get("rel") == "WORKS_ON" and ("skill", topic) in out_neighbours(source, "HAS_SKILL"):
            target_projects.add(target)

    workers = {source for source, target, data in graph.edges(data=True)
               if data.get("rel") == "WORKS_ON" and target in target_projects}

    managers = set()
    for worker in workers:
        managers.update(out_neighbours(worker, "REPORTS_TO"))

    return sorted({graph.nodes[m]["name"] for m in managers
                   if graph.nodes[m]["is_engineering"]})
```

---

### 4. Test Cases

**in1.txt**

```
1
10
{"id": 1, "name": "Nadia Osei", "manager_id": null, "is_engineering": 0, "skills": []}
{"id": 2, "name": "Priya Raman", "manager_id": 1, "is_engineering": 1, "skills": []}
{"id": 3, "name": "Marco Silva", "manager_id": 2, "is_engineering": 1, "skills": []}
{"id": 4, "name": "Elena Duarte", "manager_id": 3, "is_engineering": 1, "skills": ["Leadership", "Python"]}
{"id": 5, "name": "Sofia Klein", "manager_id": 2, "is_engineering": 0, "skills": []}
{"id": 7, "name": "Ana Iyer", "manager_id": 4, "is_engineering": 1, "skills": ["Python", "LLM Security"]}
{"id": 8, "name": "Ben Cho", "manager_id": 4, "is_engineering": 1, "skills": ["Rust"]}
{"id": 9, "name": "Chika Mba", "manager_id": 5, "is_engineering": 0, "skills": ["LLM Security"]}
{"id": 12, "name": "Grace Lin", "manager_id": 13, "is_engineering": 0, "skills": []}
{"id": 13, "name": "Hiro Tanaka", "manager_id": 2, "is_engineering": 0, "skills": []}
4
{"id": 101, "name": "Guardrail"}
{"id": 102, "name": "Atlas"}
{"id": 103, "name": "Prism"}
{"id": 104, "name": "Sentinel"}
5
{"employee_id": 7, "project_id": 101}
{"employee_id": 9, "project_id": 101}
{"employee_id": 8, "project_id": 104}
{"employee_id": 12, "project_id": 103}
{"employee_id": 7, "project_id": 102}
4
{"id": 201, "project_id": 101, "text": "Project Guardrail hardens the company LLM endpoints against prompt injection. This is core LLM Security work."}
{"id": 202, "project_id": 104, "text": "Sentinel provides continuous LLM Security monitoring of production model traffic."}
{"id": 203, "project_id": 102, "text": "Atlas is the unified data platform providing feature storage and batch pipelines."}
{"id": 205, "project_id": 103, "text": "Prism is the shared design system defining components and typography."}
LLM Security
```

**out1.txt**

```
['Elena Duarte']
```

**in2.txt**

```
1
9
{"id": 1, "name": "Nadia Osei", "manager_id": null, "is_engineering": 0, "skills": []}
{"id": 2, "name": "Priya Raman", "manager_id": 1, "is_engineering": 1, "skills": []}
{"id": 3, "name": "Marco Silva", "manager_id": 2, "is_engineering": 1, "skills": []}
{"id": 4, "name": "Elena Duarte", "manager_id": 3, "is_engineering": 1, "skills": ["Leadership"]}
{"id": 6, "name": "Tomas Berg", "manager_id": 2, "is_engineering": 0, "skills": []}
{"id": 8, "name": "Ben Cho", "manager_id": 4, "is_engineering": 1, "skills": ["Rust"]}
{"id": 10, "name": "Diego Rossi", "manager_id": 6, "is_engineering": 0, "skills": ["Python", "Embeddings"]}
{"id": 15, "name": "Omar Farouk", "manager_id": 3, "is_engineering": 1, "skills": []}
{"id": 16, "name": "Iris Wong", "manager_id": 15, "is_engineering": 1, "skills": ["Go", "Kubernetes"]}
3
{"id": 102, "name": "Atlas"}
{"id": 105, "name": "Nimbus"}
{"id": 106, "name": "Beacon"}
3
{"employee_id": 10, "project_id": 102}
{"employee_id": 16, "project_id": 105}
{"employee_id": 8, "project_id": 106}
3
{"id": 203, "project_id": 102, "text": "Atlas is the unified data platform providing feature storage and batch pipelines."}
{"id": 204, "project_id": 105, "text": "Nimbus manages the container platform, autoscaling, and the service mesh that hosts internal applications."}
{"id": 206, "project_id": 106, "text": "Beacon runs the Kubernetes admission controllers that enforce workload policy at deploy time."}
Kubernetes
```

**out2.txt**

```
['Elena Duarte', 'Omar Farouk']
```

**in3.txt**

```
1
8
{"id": 1, "name": "Nadia Osei", "manager_id": null, "is_engineering": 0, "skills": []}
{"id": 2, "name": "Priya Raman", "manager_id": 1, "is_engineering": 1, "skills": []}
{"id": 3, "name": "Marco Silva", "manager_id": 2, "is_engineering": 1, "skills": []}
{"id": 4, "name": "Elena Duarte", "manager_id": 3, "is_engineering": 1, "skills": []}
{"id": 5, "name": "Sofia Klein", "manager_id": 2, "is_engineering": 0, "skills": []}
{"id": 7, "name": "Ana Iyer", "manager_id": 3, "is_engineering": 1, "skills": ["Python"]}
{"id": 8, "name": "Ben Cho", "manager_id": 5, "is_engineering": 1, "skills": ["Rust"]}
{"id": 9, "name": "Chika Mba", "manager_id": 4, "is_engineering": 1, "skills": ["llm security"]}
2
{"id": 101, "name": "Guardrail"}
{"id": 102, "name": "Sentinel"}
3
{"employee_id": 7, "project_id": 101}
{"employee_id": 8, "project_id": 101}
{"employee_id": 9, "project_id": 102}
2
{"id": 201, "project_id": 101, "text": "Project guardrail hardens the company llm security posture against prompt injection."}
{"id": 202, "project_id": 102, "text": "Sentinel monitors production model traffic for adversarial prompts."}
LLM Security
```

**out3.txt**

```
['Marco Silva']
```


In [1]:
import networkx as nx


def managers_over_topic(employees: list, projects: list, assignments: list,
                        documents: list, topic: str) -> list:
    graph = nx.MultiDiGraph()

    for employee in employees:
        graph.add_node(("employee", employee["id"]), name=employee["name"],
                       is_engineering=employee["is_engineering"])
        if employee["manager_id"] is not None:
            graph.add_edge(("employee", employee["id"]), ("employee", employee["manager_id"]),
                           rel="REPORTS_TO")
        for skill in employee["skills"]:
            graph.add_node(("skill", skill), name=skill)
            graph.add_edge(("employee", employee["id"]), ("skill", skill), rel="HAS_SKILL")

    for project in projects:
        graph.add_node(("project", project["id"]), name=project["name"])

    for assignment in assignments:
        graph.add_edge(("employee", assignment["employee_id"]),
                       ("project", assignment["project_id"]), rel="WORKS_ON")

    for document in documents:
        graph.add_node(("document", document["id"]), text=document["text"])
        graph.add_edge(("document", document["id"]), ("project", document["project_id"]),
                       rel="ABOUT")

    def out_neighbours(node, rel):
        return [t for _, t, d in graph.out_edges(node, data=True) if d.get("rel") == rel]

    target_projects = set()
    for node, data in graph.nodes(data=True):
        if node[0] == "document" and topic.lower() in data["text"].lower():
            target_projects.update(out_neighbours(node, "ABOUT"))
    for source, target, data in graph.edges(data=True):
        if data.get("rel") == "WORKS_ON" and ("skill", topic) in out_neighbours(source, "HAS_SKILL"):
            target_projects.add(target)

    workers = {source for source, target, data in graph.edges(data=True)
               if data.get("rel") == "WORKS_ON" and target in target_projects}

    managers = set()
    for worker in workers:
        managers.update(out_neighbours(worker, "REPORTS_TO"))

    return sorted({graph.nodes[m]["name"] for m in managers
                   if graph.nodes[m]["is_engineering"]})

In [2]:
employees = [
    {"id": 1, "name": "Nadia Osei", "manager_id": None, "is_engineering": 0, "skills": []},
    {"id": 2, "name": "Priya Raman", "manager_id": 1, "is_engineering": 1, "skills": []},
    {"id": 3, "name": "Marco Silva", "manager_id": 2, "is_engineering": 1, "skills": []},
    {"id": 4, "name": "Elena Duarte", "manager_id": 3, "is_engineering": 1, "skills": ["Leadership", "Python"]},
    {"id": 5, "name": "Sofia Klein", "manager_id": 2, "is_engineering": 0, "skills": []},
    {"id": 7, "name": "Ana Iyer", "manager_id": 4, "is_engineering": 1, "skills": ["Python", "LLM Security"]},
    {"id": 8, "name": "Ben Cho", "manager_id": 4, "is_engineering": 1, "skills": ["Rust"]},
    {"id": 9, "name": "Chika Mba", "manager_id": 5, "is_engineering": 0, "skills": ["LLM Security"]},
    {"id": 12, "name": "Grace Lin", "manager_id": 13, "is_engineering": 0, "skills": []},
    {"id": 13, "name": "Hiro Tanaka", "manager_id": 2, "is_engineering": 0, "skills": []}
]
projects = [
    {"id": 101, "name": "Guardrail"},
    {"id": 102, "name": "Atlas"},
    {"id": 103, "name": "Prism"},
    {"id": 104, "name": "Sentinel"}
]
assignments = [
    {"employee_id": 7, "project_id": 101},
    {"employee_id": 9, "project_id": 101},
    {"employee_id": 8, "project_id": 104},
    {"employee_id": 12, "project_id": 103},
    {"employee_id": 7, "project_id": 102}
]
documents = [
    {"id": 201, "project_id": 101, "text": "Project Guardrail hardens the company LLM endpoints against prompt injection. This is core LLM Security work."},
    {"id": 202, "project_id": 104, "text": "Sentinel provides continuous LLM Security monitoring of production model traffic."},
    {"id": 203, "project_id": 102, "text": "Atlas is the unified data platform providing feature storage and batch pipelines."},
    {"id": 205, "project_id": 103, "text": "Prism is the shared design system defining components and typography."}
]
topic = "LLM Security"

managers_over_topic(employees, projects, assignments, documents, topic)

['Elena Duarte']

In [3]:
employees = [
    {"id": 1, "name": "Nadia Osei", "manager_id": None, "is_engineering": 0, "skills": []},
    {"id": 2, "name": "Priya Raman", "manager_id": 1, "is_engineering": 1, "skills": []},
    {"id": 3, "name": "Marco Silva", "manager_id": 2, "is_engineering": 1, "skills": []},
    {"id": 4, "name": "Elena Duarte", "manager_id": 3, "is_engineering": 1, "skills": ["Leadership"]},
    {"id": 6, "name": "Tomas Berg", "manager_id": 2, "is_engineering": 0, "skills": []},
    {"id": 8, "name": "Ben Cho", "manager_id": 4, "is_engineering": 1, "skills": ["Rust"]},
    {"id": 10, "name": "Diego Rossi", "manager_id": 6, "is_engineering": 0, "skills": ["Python", "Embeddings"]},
    {"id": 15, "name": "Omar Farouk", "manager_id": 3, "is_engineering": 1, "skills": []},
    {"id": 16, "name": "Iris Wong", "manager_id": 15, "is_engineering": 1, "skills": ["Go", "Kubernetes"]}
]
projects = [
    {"id": 102, "name": "Atlas"},
    {"id": 105, "name": "Nimbus"},
    {"id": 106, "name": "Beacon"}
]
assignments = [
    {"employee_id": 10, "project_id": 102},
    {"employee_id": 16, "project_id": 105},
    {"employee_id": 8, "project_id": 106}
]
documents = [
    {"id": 203, "project_id": 102, "text": "Atlas is the unified data platform providing feature storage and batch pipelines."},
    {"id": 204, "project_id": 105, "text": "Nimbus manages the container platform, autoscaling, and the service mesh that hosts internal applications."},
    {"id": 206, "project_id": 106, "text": "Beacon runs the Kubernetes admission controllers that enforce workload policy at deploy time."}
]
topic = "Kubernetes"

managers_over_topic(employees, projects, assignments, documents, topic)

['Elena Duarte', 'Omar Farouk']

In [ ]:
employees = [
    {"id": 1, "name": "Nadia Osei", "manager_id": None, "is_engineering": 0, "skills": []},
    {"id": 2, "name": "Priya Raman", "manager_id": 1, "is_engineering": 1, "skills": []},
    {"id": 3, "name": "Marco Silva", "manager_id": 2, "is_engineering": 1, "skills": []},
    {"id": 4, "name": "Elena Duarte", "manager_id": 3, "is_engineering": 1, "skills": []},
    {"id": 5, "name": "Sofia Klein", "manager_id": 2, "is_engineering": 0, "skills": []},
    {"id": 7, "name": "Ana Iyer", "manager_id": 3, "is_engineering": 1, "skills": ["Python"]},
    {"id": 8, "name": "Ben Cho", "manager_id": 5, "is_engineering": 1, "skills": ["Rust"]},
    {"id": 9, "name": "Chika Mba", "manager_id": 4, "is_engineering": 1, "skills": ["llm security"]},
]
projects = [
    {"id": 101, "name": "Guardrail"},
    {"id": 102, "name": "Sentinel"},
]
assignments = [
    {"employee_id": 7, "project_id": 101},
    {"employee_id": 8, "project_id": 101},
    {"employee_id": 9, "project_id": 102},
]
documents = [
    {"id": 201, "project_id": 101, "text": "Project guardrail hardens the company llm security posture against prompt injection."},
    {"id": 202, "project_id": 102, "text": "Sentinel monitors production model traffic for adversarial prompts."},
]
topic = "LLM Security"

managers_over_topic(employees, projects, assignments, documents, topic)

['Marco Silva']

### [Class 10] Question 7. Rank first, filter second

**Context:** Metadata filters and semantic ranking interact in a way that surprises people the first time
they see it. The ranking is computed over the whole collection; the filter is then applied to that
ranking and the first `k` survivors are kept. So a filtered search is not the unfiltered top-k with the
non-matching documents dropped — it walks down the global ranking
until it has enough matches, which is why a filtered result can contain a document the unfiltered top-k
never mentioned.

**Problem Statement:**

Using `sentence-transformers`, write a function
`filtered_search(titles, texts, departments, query, department, k)` that runs the same query with and
without a metadata filter.

1. Build one searchable string per document by joining its title and text as `"{title}. {text}"`.
2. Encode those strings and the query with `all-MiniLM-L6-v2` and score each document by its cosine
   similarity to the query.
3. Rank every document by that similarity, highest first.
4. Walk down that single ranking and collect two lists of titles: `"unfiltered"`, the first `k` titles
   overall, and `"filtered"`, the first `k` titles whose department equals `department`. Either list may
   end up shorter than `k` if the collection runs out.
5. Return a dictionary with exactly three keys: `"unfiltered"`, `"filtered"`, and `"top_score"` — the
   similarity of the highest-ranked document overall, rounded to 4 decimal places.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: titles (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 2: texts (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 3: departments (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 4: query (string)
    Single line: a string
  Argument 5: department (string)
    Single line: a string
  Argument 6: k (integer)
    Single line: one integer

`titles`, `texts` and `departments` have the same length, and entry `i` of each describes the same
document.
```

**Output Format:**

For each test case, print the result of `filtered_search()` on a new line.

**Sample Input:**

```
titles = ["Guardrail Overview", "Sentinel Overview", "Atlas Overview", "Nimbus Overview", "Prism Overview", "Model Input Security Policy"]
texts = [
    "Project Guardrail hardens the company LLM endpoints against prompt injection and jailbreak attacks.",
    "Sentinel provides continuous LLM security monitoring of production model traffic, detecting adversarial prompts.",
    "Atlas is the unified data platform providing feature storage, batch pipelines, and governed access to analytics tables.",
    "Nimbus manages the Kubernetes based infrastructure, autoscaling, and service mesh that host internal applications.",
    "Prism is the shared design system defining components, typography, and accessibility guidelines.",
    "Company policy requires every project that handles model inputs to complete a security threat model before launch."
]
departments = ["Security", "Security", "Data", "Engineering", "Design", "Company"]
query = "monitoring model traffic for attacks"
department = "Security"
k = 3
```

**Sample Output:**

```
{'unfiltered': ['Sentinel Overview', 'Model Input Security Policy', 'Guardrail Overview'], 'filtered': ['Sentinel Overview', 'Guardrail Overview'], 'top_score': 0.5027}
```

**Note:** Every number in the result is a plain Python `float`, not a NumPy scalar. Load the embedding
model once at module level, outside the function, so it is not reloaded on every call:

```python
EMBED_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
```

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: filtered_search
RETURN TYPE: Dictionary

ARGUMENTS:
1. titles      → List of Strings
2. texts       → List of Strings
3. departments → List of Strings
4. query       → String
5. department  → String
6. k           → Integer
```

---

### 3. Reference Solution Code

```python
import numpy as np
from sentence_transformers import SentenceTransformer

EMBED_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def filtered_search(titles: list, texts: list, departments: list, query: str,
                    department: str, k: int) -> dict:
    doc_texts = ["{}. {}".format(title, text) for title, text in zip(titles, texts)]
    matrix = np.asarray(EMBED_MODEL.encode(doc_texts, normalize_embeddings=True), dtype="float32")
    query_vec = np.asarray(EMBED_MODEL.encode([query], normalize_embeddings=True), dtype="float32")[0]

    scores = matrix @ query_vec
    ranking = np.argsort(-scores)

    unfiltered, filtered = [], []
    for i in ranking:
        if len(unfiltered) < k:
            unfiltered.append(titles[i])
        if departments[i] == department and len(filtered) < k:
            filtered.append(titles[i])
    return {"unfiltered": unfiltered, "filtered": filtered,
            "top_score": round(float(scores[ranking[0]]), 4)}
```

---

### 4. Test Cases

**in1.txt**

```
1
6
Guardrail Overview
Sentinel Overview
Atlas Overview
Nimbus Overview
Prism Overview
Model Input Security Policy
6
Project Guardrail hardens the company LLM endpoints against prompt injection and jailbreak attacks.
Sentinel provides continuous LLM security monitoring of production model traffic, detecting adversarial prompts.
Atlas is the unified data platform providing feature storage, batch pipelines, and governed access to analytics tables.
Nimbus manages the Kubernetes based infrastructure, autoscaling, and service mesh that host internal applications.
Prism is the shared design system defining components, typography, and accessibility guidelines.
Company policy requires every project that handles model inputs to complete a security threat model before launch.
6
Security
Security
Data
Engineering
Design
Company
monitoring model traffic for attacks
Security
3
```

**out1.txt**

```
{'unfiltered': ['Sentinel Overview', 'Model Input Security Policy', 'Guardrail Overview'], 'filtered': ['Sentinel Overview', 'Guardrail Overview'], 'top_score': 0.5027}
```

**in2.txt**

```
1
6
Guardrail Overview
Sentinel Overview
Atlas Overview
Nimbus Overview
Prism Overview
Model Input Security Policy
6
Project Guardrail hardens the company LLM endpoints against prompt injection and jailbreak attacks.
Sentinel provides continuous LLM security monitoring of production model traffic, detecting adversarial prompts.
Atlas is the unified data platform providing feature storage, batch pipelines, and governed access to analytics tables.
Nimbus manages the Kubernetes based infrastructure, autoscaling, and service mesh that host internal applications.
Prism is the shared design system defining components, typography, and accessibility guidelines.
Company policy requires every project that handles model inputs to complete a security threat model before launch.
6
Security
Security
Data
Engineering
Design
Company
which team owns the feature store
Design
2
```

**out2.txt**

```
{'unfiltered': ['Atlas Overview', 'Model Input Security Policy'], 'filtered': ['Prism Overview'], 'top_score': 0.2375}
```

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer

EMBED_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def filtered_search(titles: list, texts: list, departments: list, query: str,
                    department: str, k: int) -> dict:
    doc_texts = ["{}. {}".format(title, text) for title, text in zip(titles, texts)]
    matrix = np.asarray(EMBED_MODEL.encode(doc_texts, normalize_embeddings=True), dtype="float32")
    query_vec = np.asarray(EMBED_MODEL.encode([query], normalize_embeddings=True), dtype="float32")[0]

    scores = matrix @ query_vec
    ranking = np.argsort(-scores)

    unfiltered, filtered = [], []
    for i in ranking:
        if len(unfiltered) < k:
            unfiltered.append(titles[i])
        if departments[i] == department and len(filtered) < k:
            filtered.append(titles[i])
    return {"unfiltered": unfiltered, "filtered": filtered,
            "top_score": round(float(scores[ranking[0]]), 4)}

In [8]:
titles = ["Guardrail Overview", "Sentinel Overview", "Atlas Overview", "Nimbus Overview",
          "Prism Overview", "Model Input Security Policy"]
texts = [
    "Project Guardrail hardens the company LLM endpoints against prompt injection and jailbreak attacks.",
    "Sentinel provides continuous LLM security monitoring of production model traffic, detecting adversarial prompts.",
    "Atlas is the unified data platform providing feature storage, batch pipelines, and governed access to analytics tables.",
    "Nimbus manages the Kubernetes based infrastructure, autoscaling, and service mesh that host internal applications.",
    "Prism is the shared design system defining components, typography, and accessibility guidelines.",
    "Company policy requires every project that handles model inputs to complete a security threat model before launch."
]
departments = ["Security", "Security", "Data", "Engineering", "Design", "Company"]
query = "monitoring model traffic for attacks"
department = "Security"
k = 3

filtered_search(titles, texts, departments, query, department, k)

{'unfiltered': ['Sentinel Overview',
  'Model Input Security Policy',
  'Guardrail Overview'],
 'filtered': ['Sentinel Overview', 'Guardrail Overview'],
 'top_score': 0.5027}

In [9]:
titles = ["Guardrail Overview", "Sentinel Overview", "Atlas Overview", "Nimbus Overview",
          "Prism Overview", "Model Input Security Policy"]
texts = [
    "Project Guardrail hardens the company LLM endpoints against prompt injection and jailbreak attacks.",
    "Sentinel provides continuous LLM security monitoring of production model traffic, detecting adversarial prompts.",
    "Atlas is the unified data platform providing feature storage, batch pipelines, and governed access to analytics tables.",
    "Nimbus manages the Kubernetes based infrastructure, autoscaling, and service mesh that host internal applications.",
    "Prism is the shared design system defining components, typography, and accessibility guidelines.",
    "Company policy requires every project that handles model inputs to complete a security threat model before launch."
]
departments = ["Security", "Security", "Data", "Engineering", "Design", "Company"]
query = "which team owns the feature store"
department = "Design"
k = 2

filtered_search(titles, texts, departments, query, department, k)

{'unfiltered': ['Atlas Overview', 'Model Input Security Policy'],
 'filtered': ['Prism Overview'],
 'top_score': 0.2375}

### [Class 10] Question 8. The subgraph that gets serialized into the prompt

**Context:** Grounding generation on a graph means picking a neighbourhood, not a document. Starting from
a few seed entities, a bounded walk collects everything within a few hops, and the edges among the
collected nodes are serialized as relationship triples that go into the prompt. The bound is what keeps
the prompt finite, and it has a visible consequence: an edge is only serialized when **both** of its
endpoints made it into the neighbourhood, so the last ring of the walk contributes nodes without
contributing all of their edges.

**Problem Statement:**

Using NetworkX, write a function `subgraph_context(employees, projects, assignments, seeds, hops)` that
collects a bounded neighbourhood and serializes it.

1. Build a `MultiDiGraph` whose nodes are `("employee", id)` and `("project", id)` tuples, each carrying
   its `name` as a node attribute, and whose edges carry the relationship name on an attribute called
   `rel`:
   - `("employee", id)` → `("employee", manager_id)` with `rel="REPORTS_TO"`, unless the employee's
     `manager_id` is null
   - `("employee", employee_id)` → `("project", project_id)` with `rel="WORKS_ON"`, one per assignment
2. Resolve each name in `seeds` to its node — every seed name resolves to exactly one node — and mark
   those nodes visited.
3. Expand `hops` times. The first round starts from the seed nodes and each later round starts from the
   nodes first visited in the round before it; every round follows **both** outgoing and incoming edges,
   marking anything not yet visited as visited.
4. Serialize every edge whose source **and** target are both visited, as
   `(source name) -[:REL]-> (target name)`.
5. Return a dictionary with exactly two keys: `"n_visited"`, the number of visited nodes, and
   `"triples"`, the serialized edges deduplicated and sorted ascending.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: employees (list_dict)
    Line 1: N (number of employees)
    Next N lines: one JSON object per line, each with an "id", "name" and "manager_id" key
  Argument 2: projects (list_dict)
    Line 1: N (number of projects)
    Next N lines: one JSON object per line, each with an "id" and "name" key
  Argument 3: assignments (list_dict)
    Line 1: N (number of assignments)
    Next N lines: one JSON object per line, each with an "employee_id" and "project_id" key
  Argument 4: seeds (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 5: hops (integer)
    Single line: one integer

Every non-null `manager_id` names an employee in `employees`, every `project_id` names a project
in `projects`, and no two nodes share a `name`. An employee id and a project id can be the same
number, and the same employee can appear in `assignments` for the same project more than once.
```

**Output Format:**

For each test case, print the result of `subgraph_context()` on a new line.

**Sample Input:**

```
employees = [
    {"id": 2, "name": "Priya Raman", "manager_id": None},
    {"id": 3, "name": "Marco Silva", "manager_id": 2},
    {"id": 4, "name": "Elena Duarte", "manager_id": 3},
    {"id": 7, "name": "Ana Iyer", "manager_id": 4},
    {"id": 8, "name": "Ben Cho", "manager_id": 4},
    {"id": 12, "name": "Grace Lin", "manager_id": 13},
    {"id": 13, "name": "Hiro Tanaka", "manager_id": 2}
]
projects = [
    {"id": 101, "name": "Guardrail"},
    {"id": 103, "name": "Prism"}
]
assignments = [
    {"employee_id": 7, "project_id": 101},
    {"employee_id": 8, "project_id": 101},
    {"employee_id": 12, "project_id": 103}
]
seeds = ["Guardrail"]
hops = 2
```

**Sample Output:**

```
{'n_visited': 4, 'triples': ['(Ana Iyer) -[:REPORTS_TO]-> (Elena Duarte)', '(Ana Iyer) -[:WORKS_ON]-> (Guardrail)', '(Ben Cho) -[:REPORTS_TO]-> (Elena Duarte)', '(Ben Cho) -[:WORKS_ON]-> (Guardrail)']}
```

**Note:** A seed can name a project or a person; both kinds of node carry a `name`. Employee ids and
project ids are drawn from different ranges here, but the node keys are what keep the two kinds apart.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: subgraph_context
RETURN TYPE: Dictionary

ARGUMENTS:
1. employees   → List of Dicts
2. projects    → List of Dicts
3. assignments → List of Dicts
4. seeds       → List of Strings
5. hops        → Integer
```

---

### 3. Reference Solution Code

```python
import networkx as nx


def subgraph_context(employees: list, projects: list, assignments: list,
                     seeds: list, hops: int) -> dict:
    graph = nx.MultiDiGraph()

    for employee in employees:
        graph.add_node(("employee", employee["id"]), name=employee["name"])
        if employee["manager_id"] is not None:
            graph.add_edge(("employee", employee["id"]), ("employee", employee["manager_id"]),
                           rel="REPORTS_TO")
    for project in projects:
        graph.add_node(("project", project["id"]), name=project["name"])
    for assignment in assignments:
        graph.add_edge(("employee", assignment["employee_id"]),
                       ("project", assignment["project_id"]), rel="WORKS_ON")

    by_name = {data["name"]: node for node, data in graph.nodes(data=True)}
    visited = {by_name[name] for name in seeds}
    frontier = list(visited)
    for _ in range(hops):
        next_frontier = []
        for node in frontier:
            for _, target, _data in graph.out_edges(node, data=True):
                if target not in visited:
                    visited.add(target)
                    next_frontier.append(target)
            for source, _, _data in graph.in_edges(node, data=True):
                if source not in visited:
                    visited.add(source)
                    next_frontier.append(source)
        frontier = next_frontier

    triples = set()
    for source, target, data in graph.edges(data=True):
        if source in visited and target in visited:
            triples.add("({}) -[:{}]-> ({})".format(
                graph.nodes[source]["name"], data["rel"], graph.nodes[target]["name"]))

    return {"n_visited": len(visited), "triples": sorted(triples)}
```

---

### 4. Test Cases

**in1.txt**

```
1
7
{"id": 2, "name": "Priya Raman", "manager_id": null}
{"id": 3, "name": "Marco Silva", "manager_id": 2}
{"id": 4, "name": "Elena Duarte", "manager_id": 3}
{"id": 7, "name": "Ana Iyer", "manager_id": 4}
{"id": 8, "name": "Ben Cho", "manager_id": 4}
{"id": 12, "name": "Grace Lin", "manager_id": 13}
{"id": 13, "name": "Hiro Tanaka", "manager_id": 2}
2
{"id": 101, "name": "Guardrail"}
{"id": 103, "name": "Prism"}
3
{"employee_id": 7, "project_id": 101}
{"employee_id": 8, "project_id": 101}
{"employee_id": 12, "project_id": 103}
1
Guardrail
2
```

**out1.txt**

```
{'n_visited': 4, 'triples': ['(Ana Iyer) -[:REPORTS_TO]-> (Elena Duarte)', '(Ana Iyer) -[:WORKS_ON]-> (Guardrail)', '(Ben Cho) -[:REPORTS_TO]-> (Elena Duarte)', '(Ben Cho) -[:WORKS_ON]-> (Guardrail)']}
```

**in2.txt**

```
1
7
{"id": 2, "name": "Priya Raman", "manager_id": null}
{"id": 3, "name": "Marco Silva", "manager_id": 2}
{"id": 4, "name": "Elena Duarte", "manager_id": 3}
{"id": 7, "name": "Ana Iyer", "manager_id": 4}
{"id": 8, "name": "Ben Cho", "manager_id": 4}
{"id": 12, "name": "Grace Lin", "manager_id": 13}
{"id": 13, "name": "Hiro Tanaka", "manager_id": 2}
2
{"id": 101, "name": "Guardrail"}
{"id": 13, "name": "Prism"}
4
{"employee_id": 7, "project_id": 101}
{"employee_id": 7, "project_id": 101}
{"employee_id": 8, "project_id": 101}
{"employee_id": 12, "project_id": 13}
2
Grace Lin
Guardrail
1
```

**out2.txt**

```
{'n_visited': 6, 'triples': ['(Ana Iyer) -[:WORKS_ON]-> (Guardrail)', '(Ben Cho) -[:WORKS_ON]-> (Guardrail)', '(Grace Lin) -[:REPORTS_TO]-> (Hiro Tanaka)', '(Grace Lin) -[:WORKS_ON]-> (Prism)']}
```

In [10]:
import networkx as nx


def subgraph_context(employees: list, projects: list, assignments: list,
                     seeds: list, hops: int) -> dict:
    graph = nx.MultiDiGraph()

    for employee in employees:
        graph.add_node(("employee", employee["id"]), name=employee["name"])
        if employee["manager_id"] is not None:
            graph.add_edge(("employee", employee["id"]), ("employee", employee["manager_id"]),
                           rel="REPORTS_TO")
    for project in projects:
        graph.add_node(("project", project["id"]), name=project["name"])
    for assignment in assignments:
        graph.add_edge(("employee", assignment["employee_id"]),
                       ("project", assignment["project_id"]), rel="WORKS_ON")

    by_name = {data["name"]: node for node, data in graph.nodes(data=True)}
    visited = {by_name[name] for name in seeds}
    frontier = list(visited)
    for _ in range(hops):
        next_frontier = []
        for node in frontier:
            for _, target, _data in graph.out_edges(node, data=True):
                if target not in visited:
                    visited.add(target)
                    next_frontier.append(target)
            for source, _, _data in graph.in_edges(node, data=True):
                if source not in visited:
                    visited.add(source)
                    next_frontier.append(source)
        frontier = next_frontier

    triples = set()
    for source, target, data in graph.edges(data=True):
        if source in visited and target in visited:
            triples.add("({}) -[:{}]-> ({})".format(
                graph.nodes[source]["name"], data["rel"], graph.nodes[target]["name"]))

    return {"n_visited": len(visited), "triples": sorted(triples)}

In [11]:
employees = [
    {"id": 2, "name": "Priya Raman", "manager_id": None},
    {"id": 3, "name": "Marco Silva", "manager_id": 2},
    {"id": 4, "name": "Elena Duarte", "manager_id": 3},
    {"id": 7, "name": "Ana Iyer", "manager_id": 4},
    {"id": 8, "name": "Ben Cho", "manager_id": 4},
    {"id": 12, "name": "Grace Lin", "manager_id": 13},
    {"id": 13, "name": "Hiro Tanaka", "manager_id": 2}
]
projects = [
    {"id": 101, "name": "Guardrail"},
    {"id": 103, "name": "Prism"}
]
assignments = [
    {"employee_id": 7, "project_id": 101},
    {"employee_id": 8, "project_id": 101},
    {"employee_id": 12, "project_id": 103}
]
seeds = ["Guardrail"]
hops = 2

subgraph_context(employees, projects, assignments, seeds, hops)

{'n_visited': 4,
 'triples': ['(Ana Iyer) -[:REPORTS_TO]-> (Elena Duarte)',
  '(Ana Iyer) -[:WORKS_ON]-> (Guardrail)',
  '(Ben Cho) -[:REPORTS_TO]-> (Elena Duarte)',
  '(Ben Cho) -[:WORKS_ON]-> (Guardrail)']}

In [12]:
employees = [
    {"id": 2, "name": "Priya Raman", "manager_id": None},
    {"id": 3, "name": "Marco Silva", "manager_id": 2},
    {"id": 4, "name": "Elena Duarte", "manager_id": 3},
    {"id": 7, "name": "Ana Iyer", "manager_id": 4},
    {"id": 8, "name": "Ben Cho", "manager_id": 4},
    {"id": 12, "name": "Grace Lin", "manager_id": 13},
    {"id": 13, "name": "Hiro Tanaka", "manager_id": 2}
]
projects = [
    {"id": 101, "name": "Guardrail"},
    {"id": 13, "name": "Prism"}
]
assignments = [
    {"employee_id": 7, "project_id": 101},
    {"employee_id": 7, "project_id": 101},
    {"employee_id": 8, "project_id": 101},
    {"employee_id": 12, "project_id": 13}
]
seeds = ["Grace Lin", "Guardrail"]
hops = 1

subgraph_context(employees, projects, assignments, seeds, hops)

{'n_visited': 6, 'triples': ['(Ana Iyer) -[:WORKS_ON]-> (Guardrail)', '(Ben Cho) -[:WORKS_ON]-> (Guardrail)', '(Grace Lin) -[:REPORTS_TO]-> (Hiro Tanaka)', '(Grace Lin) -[:WORKS_ON]-> (Prism)']}


### [Class 11] Question 9. Fusing a lexical and a dense ranking

**Context:** Lexical scoring and dense scoring fail in opposite directions. BM25 nails an exact
identifier like `INC-2847` and goes blind the moment a user paraphrases; embeddings handle the
paraphrase and smear across near-identical identifiers. Their scores are not on a comparable scale, so
they cannot simply be added. Reciprocal rank fusion sidesteps that entirely by throwing the scores away
and combining positions instead: each list contributes `1 / (rrf_k + rank)` to every document it ranked,
and the constant `rrf_k` decides how much a top position is worth relative to a merely-present one.

**Problem Statement:**

Write a function `hybrid_rank(doc_texts, query, shortlist, k, rrf_k)` that fuses a BM25 ranking and a
dense ranking of the same corpus.

1. Encode `doc_texts` and `query` with `all-MiniLM-L6-v2`, asking the model for normalized embeddings,
   and rank the documents by their dot-product similarity. Keep the top `shortlist` positions.
2. Score the same corpus with `BM25Okapi`, comparing documents and query on the same terms: lowercase
   the text and treat every run of whitespace or punctuation as a term boundary, except a hyphen inside
   an identifier such as `INC-2847` or `red-team`, and remove no stopwords. Keep the top `shortlist`
   positions.
   In either ranking, when two documents score exactly the same, the one with the lower
   document index ranks first.
3. Fuse the two rankings: each document accumulates `1 / (rrf_k + rank)` from each list that contains it,
   where `rank` is 1-based.
4. Return a dictionary mapping the 0-based index of each of the top `k` fused documents to its fused
   score rounded to 4 decimal places, ordered by fused score descending. When two documents end up
   with exactly the same fused score, the one that appeared earlier in the BM25 ranking ranks first,
   and a document the BM25 ranking did not contain ranks after one it did.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: doc_texts (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 2: query (string)
    Single line: a string
  Argument 3: shortlist (integer)
    Single line: one integer
  Argument 4: k (integer)
    Single line: one integer
  Argument 5: rrf_k (integer)
    Single line: one integer
```

**Output Format:**

For each test case, print the result of `hybrid_rank()` on a new line.

**Sample Input:**

```
doc_texts = [
    "Guardrail Overview. Project Guardrail hardens the company LLM endpoints against prompt injection and jailbreak attacks.",
    "INC-2847 Postmortem. Incident INC-2847 involved a prompt injection attack against the Guardrail-protected chat endpoint.",
    "Atlas Overview. Atlas is the unified data platform providing feature storage and batch pipelines.",
    "Data Retention Policy. Personal data is retained for at most 90 days in operational stores.",
    "New Hire Onboarding Guide. New engineers are paired with a buddy for the first two weeks."
]
query = "the INC-2847 postmortem on the chat endpoint"
shortlist = 3
k = 3
rrf_k = 60
```

**Sample Output:**

```
{1: 0.0328, 0: 0.032, 2: 0.0161}
```

**Note:** The dictionary's keys and values are plain Python numbers, not NumPy scalars. A document
that only one of the two rankings shortlisted still earns its contribution from that one list. Load the
embedding model once at module level, outside the function:

```python
EMBED_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
```

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: hybrid_rank
RETURN TYPE: Dictionary

ARGUMENTS:
1. doc_texts → List of Strings
2. query     → String
3. shortlist → Integer
4. k         → Integer
5. rrf_k     → Integer
```

---

### 3. Reference Solution Code

```python
import re

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

EMBED_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def tokenize(text: str) -> list:
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())


def hybrid_rank(doc_texts: list, query: str, shortlist: int, k: int, rrf_k: int) -> dict:
    matrix = np.asarray(EMBED_MODEL.encode(doc_texts, normalize_embeddings=True), dtype="float32")
    query_vec = np.asarray(EMBED_MODEL.encode([query], normalize_embeddings=True), dtype="float32")[0]
    dense_ids = list(np.argsort(-(matrix @ query_vec), kind="stable")[:shortlist])

    bm25 = BM25Okapi([tokenize(text) for text in doc_texts])
    bm25_ids = list(np.argsort(-bm25.get_scores(tokenize(query)), kind="stable")[:shortlist])

    fused = {}
    for ranking in (bm25_ids, dense_ids):
        for rank, doc_id in enumerate(ranking, start=1):
            fused[int(doc_id)] = fused.get(int(doc_id), 0.0) + 1.0 / (rrf_k + rank)

    ordered = sorted(fused.items(), key=lambda pair: -pair[1])[:k]
    return {doc_id: round(score, 4) for doc_id, score in ordered}
```

---

### 4. Test Cases

**in1.txt**

```
1
12
Guardrail Overview. Project Guardrail hardens the company LLM endpoints against prompt injection and jailbreak attacks. It covers input sanitization, output filtering, and periodic red teaming.
Sentinel Overview. Sentinel provides continuous LLM security monitoring of production model traffic, detecting adversarial prompts and potential data exfiltration in real time.
Atlas Overview. Atlas is the unified data platform providing feature storage, batch pipelines, and governed access to analytics tables across the company.
Nimbus Overview. Nimbus manages the Kubernetes based infrastructure, autoscaling, and service mesh that host internal applications.
INC-2847 Postmortem. Incident INC-2847 involved a prompt injection attack against the Guardrail-protected chat endpoint. Root cause: output filter regression after the v2.3 deploy.
INC-3102 Postmortem. Incident INC-3102 was a Kubernetes autoscaler misconfiguration in Nimbus that caused cascading pod evictions during peak traffic.
RFC-014 Cross-Encoder Reranking. This RFC proposes adopting cross-encoder reranking on top of hybrid retrieval in the internal assistant. Recommended shortlist size: 20.
Data Retention Policy. Personal data is retained for at most 90 days in operational stores. Aggregated analytics data may be retained for 24 months.
How Guardrail Handles Jailbreaks. The defense pipeline layers instruction hierarchy tagging, adversarial suffix detection, and periodic red-team evaluations against a curated attack set.
Atlas Feature Store Internals. Feature values in Atlas are versioned by ingestion timestamp. Point-in-time joins use a snapshot table to prevent training-serving skew.
Nimbus Service Mesh Design. Nimbus uses a sidecar service mesh for mTLS and traffic shaping. Circuit breakers open on 5xx rates exceeding 5 percent for 30 seconds.
New Hire Onboarding Guide. New engineers are paired with a buddy for the first two weeks, complete the security training module, and ship a small starter task by end of week two.
what was the root cause in the INC-2847 postmortem
5
4
60
```

**out1.txt**

```
{4: 0.0328, 5: 0.0323, 6: 0.0315, 2: 0.0159}
```

**in2.txt**

```
1
12
Guardrail Overview. Project Guardrail hardens the company LLM endpoints against prompt injection and jailbreak attacks. It covers input sanitization, output filtering, and periodic red teaming.
Sentinel Overview. Sentinel provides continuous LLM security monitoring of production model traffic, detecting adversarial prompts and potential data exfiltration in real time.
Atlas Overview. Atlas is the unified data platform providing feature storage, batch pipelines, and governed access to analytics tables across the company.
Nimbus Overview. Nimbus manages the Kubernetes based infrastructure, autoscaling, and service mesh that host internal applications.
INC-2847 Postmortem. Incident INC-2847 involved a prompt injection attack against the Guardrail-protected chat endpoint. Root cause: output filter regression after the v2.3 deploy.
INC-3102 Postmortem. Incident INC-3102 was a Kubernetes autoscaler misconfiguration in Nimbus that caused cascading pod evictions during peak traffic.
RFC-014 Cross-Encoder Reranking. This RFC proposes adopting cross-encoder reranking on top of hybrid retrieval in the internal assistant. Recommended shortlist size: 20.
Data Retention Policy. Personal data is retained for at most 90 days in operational stores. Aggregated analytics data may be retained for 24 months.
How Guardrail Handles Jailbreaks. The defense pipeline layers instruction hierarchy tagging, adversarial suffix detection, and periodic red-team evaluations against a curated attack set.
Atlas Feature Store Internals. Feature values in Atlas are versioned by ingestion timestamp. Point-in-time joins use a snapshot table to prevent training-serving skew.
Nimbus Service Mesh Design. Nimbus uses a sidecar service mesh for mTLS and traffic shaping. Circuit breakers open on 5xx rates exceeding 5 percent for 30 seconds.
New Hire Onboarding Guide. New engineers are paired with a buddy for the first two weeks, complete the security training module, and ship a small starter task by end of week two.
how do we stop attackers from tricking the chatbot with clever prompts
5
3
10
```

**out2.txt**

```
{8: 0.1742, 4: 0.1576, 1: 0.1538}
```

In [13]:
import re

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

EMBED_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def tokenize(text: str) -> list:
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())


def hybrid_rank(doc_texts: list, query: str, shortlist: int, k: int, rrf_k: int) -> dict:
    matrix = np.asarray(EMBED_MODEL.encode(doc_texts, normalize_embeddings=True), dtype="float32")
    query_vec = np.asarray(EMBED_MODEL.encode([query], normalize_embeddings=True), dtype="float32")[0]
    dense_ids = list(np.argsort(-(matrix @ query_vec), kind="stable")[:shortlist])

    bm25 = BM25Okapi([tokenize(text) for text in doc_texts])
    bm25_ids = list(np.argsort(-bm25.get_scores(tokenize(query)), kind="stable")[:shortlist])

    fused = {}
    for ranking in (bm25_ids, dense_ids):
        for rank, doc_id in enumerate(ranking, start=1):
            fused[int(doc_id)] = fused.get(int(doc_id), 0.0) + 1.0 / (rrf_k + rank)

    ordered = sorted(fused.items(), key=lambda pair: -pair[1])[:k]
    return {doc_id: round(score, 4) for doc_id, score in ordered}

In [14]:
doc_texts = [
    "Guardrail Overview. Project Guardrail hardens the company LLM endpoints against prompt injection and jailbreak attacks. It covers input sanitization, output filtering, and periodic red teaming.",
    "Sentinel Overview. Sentinel provides continuous LLM security monitoring of production model traffic, detecting adversarial prompts and potential data exfiltration in real time.",
    "Atlas Overview. Atlas is the unified data platform providing feature storage, batch pipelines, and governed access to analytics tables across the company.",
    "Nimbus Overview. Nimbus manages the Kubernetes based infrastructure, autoscaling, and service mesh that host internal applications.",
    "INC-2847 Postmortem. Incident INC-2847 involved a prompt injection attack against the Guardrail-protected chat endpoint. Root cause: output filter regression after the v2.3 deploy.",
    "INC-3102 Postmortem. Incident INC-3102 was a Kubernetes autoscaler misconfiguration in Nimbus that caused cascading pod evictions during peak traffic.",
    "RFC-014 Cross-Encoder Reranking. This RFC proposes adopting cross-encoder reranking on top of hybrid retrieval in the internal assistant. Recommended shortlist size: 20.",
    "Data Retention Policy. Personal data is retained for at most 90 days in operational stores. Aggregated analytics data may be retained for 24 months.",
    "How Guardrail Handles Jailbreaks. The defense pipeline layers instruction hierarchy tagging, adversarial suffix detection, and periodic red-team evaluations against a curated attack set.",
    "Atlas Feature Store Internals. Feature values in Atlas are versioned by ingestion timestamp. Point-in-time joins use a snapshot table to prevent training-serving skew.",
    "Nimbus Service Mesh Design. Nimbus uses a sidecar service mesh for mTLS and traffic shaping. Circuit breakers open on 5xx rates exceeding 5 percent for 30 seconds.",
    "New Hire Onboarding Guide. New engineers are paired with a buddy for the first two weeks, complete the security training module, and ship a small starter task by end of week two."
]
query = "what was the root cause in the INC-2847 postmortem"
shortlist = 5
k = 4
rrf_k = 60

hybrid_rank(doc_texts, query, shortlist, k, rrf_k)

{4: 0.0328, 5: 0.0323, 6: 0.0315, 2: 0.0159}

In [15]:
doc_texts = [
    "Guardrail Overview. Project Guardrail hardens the company LLM endpoints against prompt injection and jailbreak attacks. It covers input sanitization, output filtering, and periodic red teaming.",
    "Sentinel Overview. Sentinel provides continuous LLM security monitoring of production model traffic, detecting adversarial prompts and potential data exfiltration in real time.",
    "Atlas Overview. Atlas is the unified data platform providing feature storage, batch pipelines, and governed access to analytics tables across the company.",
    "Nimbus Overview. Nimbus manages the Kubernetes based infrastructure, autoscaling, and service mesh that host internal applications.",
    "INC-2847 Postmortem. Incident INC-2847 involved a prompt injection attack against the Guardrail-protected chat endpoint. Root cause: output filter regression after the v2.3 deploy.",
    "INC-3102 Postmortem. Incident INC-3102 was a Kubernetes autoscaler misconfiguration in Nimbus that caused cascading pod evictions during peak traffic.",
    "RFC-014 Cross-Encoder Reranking. This RFC proposes adopting cross-encoder reranking on top of hybrid retrieval in the internal assistant. Recommended shortlist size: 20.",
    "Data Retention Policy. Personal data is retained for at most 90 days in operational stores. Aggregated analytics data may be retained for 24 months.",
    "How Guardrail Handles Jailbreaks. The defense pipeline layers instruction hierarchy tagging, adversarial suffix detection, and periodic red-team evaluations against a curated attack set.",
    "Atlas Feature Store Internals. Feature values in Atlas are versioned by ingestion timestamp. Point-in-time joins use a snapshot table to prevent training-serving skew.",
    "Nimbus Service Mesh Design. Nimbus uses a sidecar service mesh for mTLS and traffic shaping. Circuit breakers open on 5xx rates exceeding 5 percent for 30 seconds.",
    "New Hire Onboarding Guide. New engineers are paired with a buddy for the first two weeks, complete the security training module, and ship a small starter task by end of week two."
]
query = "how do we stop attackers from tricking the chatbot with clever prompts"
shortlist = 5
k = 3
rrf_k = 10

hybrid_rank(doc_texts, query, shortlist, k, rrf_k)

{8: 0.1742, 4: 0.1576, 1: 0.1538}

### [Class 11] Question 10. Paying for precision at the last step

**Context:** Retrieval is fast because it never compares a query and a document directly — both are
encoded separately and matched by geometry. A cross-encoder does the expensive thing instead: it reads
the query and one document together and scores that pair. It is far too slow to run over a corpus, so it
runs over a shortlist that cheap retrieval has already produced, and its job is to reorder the last few
candidates that cheap retrieval got in roughly the right neighbourhood but the wrong order.

**Problem Statement:**

Write a function `rerank_shortlist(doc_texts, query, shortlist, top_k)` that produces a fused shortlist
and then reorders it with a cross-encoder.

1. Produce a hybrid shortlist exactly as reciprocal rank fusion prescribes: rank the corpus with
   `all-MiniLM-L6-v2` normalized embeddings by dot product, rank it again with `BM25Okapi` over the
   same terms — lowercase the text and treat every run of whitespace or punctuation as a term boundary,
   except a hyphen inside an identifier such as `INC-2847` or `red-team`, and remove no stopwords —
   keep the top `shortlist` positions of each, and fuse them with `1 / (60 + rank)`, where `rank` is 1-based. Take the
   `shortlist` best-fused documents, ordered by fused score descending.
   In either ranking, when two documents score exactly the same, the one with the lower
   document index ranks first; when two documents end up with exactly the same **fused** score, the
   one that appeared earlier in the BM25 ranking ranks first, and a document the BM25 ranking did
   not contain ranks after one it did.
2. Score every `(query, document)` pair in that shortlist with the cross-encoder
   `cross-encoder/ms-marco-MiniLM-L-6-v2`.
3. Reorder the shortlist by cross-encoder score, highest first, and keep the top `top_k`.
4. Return a dictionary with exactly these three keys, in this order: `"before"`, the first `top_k` document indexes in fused
   order; `"after"`, the `top_k` document indexes in reranked order; and `"top_score"`, the highest
   cross-encoder score, rounded to 2 decimal places.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: doc_texts (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 2: query (string)
    Single line: a string
  Argument 3: shortlist (integer)
    Single line: one integer
  Argument 4: top_k (integer)
    Single line: one integer
```

**Output Format:**

For each test case, print the result of `rerank_shortlist()` on a new line.

**Sample Input:**

```
doc_texts = [
    "Guardrail Overview. Project Guardrail hardens the company LLM endpoints against prompt injection and jailbreak attacks.",
    "INC-2847 Postmortem. Incident INC-2847 involved a prompt injection attack against the Guardrail-protected chat endpoint.",
    "Atlas Overview. Atlas is the unified data platform providing feature storage and batch pipelines.",
    "Data Retention Policy. Personal data is retained for at most 90 days in operational stores.",
    "New Hire Onboarding Guide. New engineers are paired with a buddy for the first two weeks."
]
query = "how long is the personal data kept in the operational stores"
shortlist = 4
top_k = 3
```

**Sample Output:**

```
{'before': [3, 2, 1], 'after': [3, 4, 2], 'top_score': 9.68}
```

**Note:** The document indexes and the score are plain Python numbers, not NumPy scalars.
Cross-encoder scores are logits, not probabilities: they are unbounded and frequently negative.
Return the raw score, not a normalized one. Load both models once at module level, outside the function:

```python
EMBED_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
RERANKER = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
```

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: rerank_shortlist
RETURN TYPE: Dictionary

ARGUMENTS:
1. doc_texts → List of Strings
2. query     → String
3. shortlist → Integer
4. top_k     → Integer
```

---

### 3. Reference Solution Code

```python
import re

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer

EMBED_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
RERANKER = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def tokenize(text: str) -> list:
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())


def rerank_shortlist(doc_texts: list, query: str, shortlist: int, top_k: int) -> dict:
    matrix = np.asarray(EMBED_MODEL.encode(doc_texts, normalize_embeddings=True), dtype="float32")
    query_vec = np.asarray(EMBED_MODEL.encode([query], normalize_embeddings=True), dtype="float32")[0]
    dense_ids = list(np.argsort(-(matrix @ query_vec), kind="stable")[:shortlist])

    bm25 = BM25Okapi([tokenize(text) for text in doc_texts])
    bm25_ids = list(np.argsort(-bm25.get_scores(tokenize(query)), kind="stable")[:shortlist])

    fused = {}
    for ranking in (bm25_ids, dense_ids):
        for rank, doc_id in enumerate(ranking, start=1):
            fused[int(doc_id)] = fused.get(int(doc_id), 0.0) + 1.0 / (60 + rank)
    before = [doc_id for doc_id, _ in sorted(fused.items(), key=lambda pair: -pair[1])[:shortlist]]

    scores = RERANKER.predict([(query, doc_texts[doc_id]) for doc_id in before])
    reranked = sorted(zip(before, scores), key=lambda pair: -pair[1])[:top_k]

    return {"before": before[:top_k],
            "after": [doc_id for doc_id, _ in reranked],
            "top_score": round(float(reranked[0][1]), 2)}
```

---

### 4. Test Cases

**in1.txt**

```
1
12
Guardrail Overview. Project Guardrail hardens the company LLM endpoints against prompt injection and jailbreak attacks. It covers input sanitization, output filtering, and periodic red teaming.
Sentinel Overview. Sentinel provides continuous LLM security monitoring of production model traffic, detecting adversarial prompts and potential data exfiltration in real time.
Atlas Overview. Atlas is the unified data platform providing feature storage, batch pipelines, and governed access to analytics tables across the company.
Nimbus Overview. Nimbus manages the Kubernetes based infrastructure, autoscaling, and service mesh that host internal applications.
INC-2847 Postmortem. Incident INC-2847 involved a prompt injection attack against the Guardrail-protected chat endpoint. Root cause: output filter regression after the v2.3 deploy.
INC-3102 Postmortem. Incident INC-3102 was a Kubernetes autoscaler misconfiguration in Nimbus that caused cascading pod evictions during peak traffic.
RFC-014 Cross-Encoder Reranking. This RFC proposes adopting cross-encoder reranking on top of hybrid retrieval in the internal assistant. Recommended shortlist size: 20.
Data Retention Policy. Personal data is retained for at most 90 days in operational stores. Aggregated analytics data may be retained for 24 months.
How Guardrail Handles Jailbreaks. The defense pipeline layers instruction hierarchy tagging, adversarial suffix detection, and periodic red-team evaluations against a curated attack set.
Atlas Feature Store Internals. Feature values in Atlas are versioned by ingestion timestamp. Point-in-time joins use a snapshot table to prevent training-serving skew.
Nimbus Service Mesh Design. Nimbus uses a sidecar service mesh for mTLS and traffic shaping. Circuit breakers open on 5xx rates exceeding 5 percent for 30 seconds.
New Hire Onboarding Guide. New engineers are paired with a buddy for the first two weeks, complete the security training module, and ship a small starter task by end of week two.
how do we protect the chatbot from adversarial prompts
6
3
```

**out1.txt**

```
{'before': [8, 1, 4], 'after': [4, 1, 8], 'top_score': -2.56}
```

**in2.txt**

```
1
12
Guardrail Overview. Project Guardrail hardens the company LLM endpoints against prompt injection and jailbreak attacks. It covers input sanitization, output filtering, and periodic red teaming.
Sentinel Overview. Sentinel provides continuous LLM security monitoring of production model traffic, detecting adversarial prompts and potential data exfiltration in real time.
Atlas Overview. Atlas is the unified data platform providing feature storage, batch pipelines, and governed access to analytics tables across the company.
Nimbus Overview. Nimbus manages the Kubernetes based infrastructure, autoscaling, and service mesh that host internal applications.
INC-2847 Postmortem. Incident INC-2847 involved a prompt injection attack against the Guardrail-protected chat endpoint. Root cause: output filter regression after the v2.3 deploy.
INC-3102 Postmortem. Incident INC-3102 was a Kubernetes autoscaler misconfiguration in Nimbus that caused cascading pod evictions during peak traffic.
RFC-014 Cross-Encoder Reranking. This RFC proposes adopting cross-encoder reranking on top of hybrid retrieval in the internal assistant. Recommended shortlist size: 20.
Data Retention Policy. Personal data is retained for at most 90 days in operational stores. Aggregated analytics data may be retained for 24 months.
How Guardrail Handles Jailbreaks. The defense pipeline layers instruction hierarchy tagging, adversarial suffix detection, and periodic red-team evaluations against a curated attack set.
Atlas Feature Store Internals. Feature values in Atlas are versioned by ingestion timestamp. Point-in-time joins use a snapshot table to prevent training-serving skew.
Nimbus Service Mesh Design. Nimbus uses a sidecar service mesh for mTLS and traffic shaping. Circuit breakers open on 5xx rates exceeding 5 percent for 30 seconds.
New Hire Onboarding Guide. New engineers are paired with a buddy for the first two weeks, complete the security training module, and ship a small starter task by end of week two.
what happened during the Kubernetes autoscaling outage
4
2
```

**out2.txt**

```
{'before': [3, 5], 'after': [5, 3], 'top_score': 5.23}
```

In [16]:
import re

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer

EMBED_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
RERANKER = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def tokenize(text: str) -> list:
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())


def rerank_shortlist(doc_texts: list, query: str, shortlist: int, top_k: int) -> dict:
    matrix = np.asarray(EMBED_MODEL.encode(doc_texts, normalize_embeddings=True), dtype="float32")
    query_vec = np.asarray(EMBED_MODEL.encode([query], normalize_embeddings=True), dtype="float32")[0]
    dense_ids = list(np.argsort(-(matrix @ query_vec), kind="stable")[:shortlist])

    bm25 = BM25Okapi([tokenize(text) for text in doc_texts])
    bm25_ids = list(np.argsort(-bm25.get_scores(tokenize(query)), kind="stable")[:shortlist])

    fused = {}
    for ranking in (bm25_ids, dense_ids):
        for rank, doc_id in enumerate(ranking, start=1):
            fused[int(doc_id)] = fused.get(int(doc_id), 0.0) + 1.0 / (60 + rank)
    before = [doc_id for doc_id, _ in sorted(fused.items(), key=lambda pair: -pair[1])[:shortlist]]

    scores = RERANKER.predict([(query, doc_texts[doc_id]) for doc_id in before])
    reranked = sorted(zip(before, scores), key=lambda pair: -pair[1])[:top_k]

    return {"before": before[:top_k],
            "after": [doc_id for doc_id, _ in reranked],
            "top_score": round(float(reranked[0][1]), 2)}

In [17]:
doc_texts = [
    "Guardrail Overview. Project Guardrail hardens the company LLM endpoints against prompt injection and jailbreak attacks. It covers input sanitization, output filtering, and periodic red teaming.",
    "Sentinel Overview. Sentinel provides continuous LLM security monitoring of production model traffic, detecting adversarial prompts and potential data exfiltration in real time.",
    "Atlas Overview. Atlas is the unified data platform providing feature storage, batch pipelines, and governed access to analytics tables across the company.",
    "Nimbus Overview. Nimbus manages the Kubernetes based infrastructure, autoscaling, and service mesh that host internal applications.",
    "INC-2847 Postmortem. Incident INC-2847 involved a prompt injection attack against the Guardrail-protected chat endpoint. Root cause: output filter regression after the v2.3 deploy.",
    "INC-3102 Postmortem. Incident INC-3102 was a Kubernetes autoscaler misconfiguration in Nimbus that caused cascading pod evictions during peak traffic.",
    "RFC-014 Cross-Encoder Reranking. This RFC proposes adopting cross-encoder reranking on top of hybrid retrieval in the internal assistant. Recommended shortlist size: 20.",
    "Data Retention Policy. Personal data is retained for at most 90 days in operational stores. Aggregated analytics data may be retained for 24 months.",
    "How Guardrail Handles Jailbreaks. The defense pipeline layers instruction hierarchy tagging, adversarial suffix detection, and periodic red-team evaluations against a curated attack set.",
    "Atlas Feature Store Internals. Feature values in Atlas are versioned by ingestion timestamp. Point-in-time joins use a snapshot table to prevent training-serving skew.",
    "Nimbus Service Mesh Design. Nimbus uses a sidecar service mesh for mTLS and traffic shaping. Circuit breakers open on 5xx rates exceeding 5 percent for 30 seconds.",
    "New Hire Onboarding Guide. New engineers are paired with a buddy for the first two weeks, complete the security training module, and ship a small starter task by end of week two."
]
query = "how do we protect the chatbot from adversarial prompts"
shortlist = 6
top_k = 3

rerank_shortlist(doc_texts, query, shortlist, top_k)

{'before': [8, 1, 4], 'after': [4, 1, 8], 'top_score': -2.56}

In [18]:
doc_texts = [
    "Guardrail Overview. Project Guardrail hardens the company LLM endpoints against prompt injection and jailbreak attacks. It covers input sanitization, output filtering, and periodic red teaming.",
    "Sentinel Overview. Sentinel provides continuous LLM security monitoring of production model traffic, detecting adversarial prompts and potential data exfiltration in real time.",
    "Atlas Overview. Atlas is the unified data platform providing feature storage, batch pipelines, and governed access to analytics tables across the company.",
    "Nimbus Overview. Nimbus manages the Kubernetes based infrastructure, autoscaling, and service mesh that host internal applications.",
    "INC-2847 Postmortem. Incident INC-2847 involved a prompt injection attack against the Guardrail-protected chat endpoint. Root cause: output filter regression after the v2.3 deploy.",
    "INC-3102 Postmortem. Incident INC-3102 was a Kubernetes autoscaler misconfiguration in Nimbus that caused cascading pod evictions during peak traffic.",
    "RFC-014 Cross-Encoder Reranking. This RFC proposes adopting cross-encoder reranking on top of hybrid retrieval in the internal assistant. Recommended shortlist size: 20.",
    "Data Retention Policy. Personal data is retained for at most 90 days in operational stores. Aggregated analytics data may be retained for 24 months.",
    "How Guardrail Handles Jailbreaks. The defense pipeline layers instruction hierarchy tagging, adversarial suffix detection, and periodic red-team evaluations against a curated attack set.",
    "Atlas Feature Store Internals. Feature values in Atlas are versioned by ingestion timestamp. Point-in-time joins use a snapshot table to prevent training-serving skew.",
    "Nimbus Service Mesh Design. Nimbus uses a sidecar service mesh for mTLS and traffic shaping. Circuit breakers open on 5xx rates exceeding 5 percent for 30 seconds.",
    "New Hire Onboarding Guide. New engineers are paired with a buddy for the first two weeks, complete the security training module, and ship a small starter task by end of week two."
]
query = "what happened during the Kubernetes autoscaling outage"
shortlist = 4
top_k = 2

rerank_shortlist(doc_texts, query, shortlist, top_k)

{'before': [3, 5], 'after': [5, 3], 'top_score': 5.23}

### [Class 11] Question 11. Measuring which retriever is actually better

**Context:** Arguments about lexical versus dense retrieval are settled by a labelled query set, not by
intuition. Two numbers carry most of the signal. Hit rate at 3 asks whether the correct document reached
a position a human would plausibly read. Mean reciprocal rank asks *where* it landed, scoring a
first-place result at 1, a second-place result at 0.5, and a miss at 0 — so two retrievers can tie on hit
rate while one of them is consistently putting the answer one line lower.

**Problem Statement:**

Write a function `compare_retrievers(doc_texts, queries, targets, k)` that evaluates a BM25 retriever and
a dense retriever on the same labelled queries.

1. Build both retrievers over `doc_texts`: `BM25Okapi` over the same terms from the documents and each
   query — lowercase the text and treat every run of whitespace or punctuation as a term boundary,
   except a hyphen inside an identifier such as `INC-2847` or `red-team`, and remove no stopwords — and
   dense search with `all-MiniLM-L6-v2` normalized embeddings ranked by dot product.
   In either ranking, when two documents score exactly the same, the one with the lower
   document index ranks first.
2. For each query, take each retriever's top `k` document indexes. `targets[i]` is a 0-based index into `doc_texts`; the 1-based
   position of that query's target document in that list is its rank; if the target is not in the top `k`, the rank is `None`.
3. For each retriever compute the hit rate at 3 — the share of queries whose rank is 3 or better, whatever `k` is — and
   the mean reciprocal rank, where a query contributes `1 / rank`, or `0.0` when the rank is `None`.
4. Return a dictionary with exactly these six keys, in this order: `"bm25_hit@3"`, `"bm25_mrr"`, `"bm25_ranks"`,
   `"dense_hit@3"`, `"dense_mrr"`, `"dense_ranks"`. The four numeric metrics are rounded to 3 decimal
   places; each `_ranks` value is the list of per-query ranks in query order.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: doc_texts (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 2: queries (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 3: targets (list_int)
    Line 1: N (list length)
    Next N lines: one integer each
  Argument 4: k (integer)
    Single line: one integer
```

**Output Format:**

For each test case, print the result of `compare_retrievers()` on a new line.

**Sample Input:**

```
doc_texts = [
    "Guardrail Overview. Project Guardrail hardens the company LLM endpoints against prompt injection and jailbreak attacks.",
    "INC-2847 Postmortem. Incident INC-2847 involved a prompt injection attack against the Guardrail-protected chat endpoint.",
    "Atlas Overview. Atlas is the unified data platform providing feature storage and batch pipelines.",
    "Data Retention Policy. Personal data is retained for at most 90 days in operational stores.",
    "New Hire Onboarding Guide. New engineers are paired with a buddy for the first two weeks."
]
queries = ["when do we throw away the records of a customer", "prompt injection attack on the chat endpoint"]
targets = [3, 1]
k = 3
```

**Sample Output:**

```
{'bm25_hit@3': 0.5, 'bm25_mrr': 0.5, 'bm25_ranks': [None, 1], 'dense_hit@3': 1.0, 'dense_mrr': 1.0, 'dense_ranks': [1, 1]}
```

**Note:** Every metric and rank in the result is a plain Python number, not a NumPy scalar. A missed
query still counts in the denominator of both metrics. Load the embedding model once at module level,
outside the function:

```python
EMBED_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
```

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: compare_retrievers
RETURN TYPE: Dictionary

ARGUMENTS:
1. doc_texts → List of Strings
2. queries   → List of Strings
3. targets   → List of Ints
4. k         → Integer
```

---

### 3. Reference Solution Code

```python
import re

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

EMBED_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def tokenize(text: str) -> list:
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())


def compare_retrievers(doc_texts: list, queries: list, targets: list, k: int) -> dict:
    matrix = np.asarray(EMBED_MODEL.encode(doc_texts, normalize_embeddings=True), dtype="float32")
    bm25 = BM25Okapi([tokenize(text) for text in doc_texts])

    def dense_search(query):
        query_vec = np.asarray(EMBED_MODEL.encode([query], normalize_embeddings=True), dtype="float32")[0]
        return [int(i) for i in np.argsort(-(matrix @ query_vec), kind="stable")[:k]]

    def bm25_search(query):
        return [int(i) for i in np.argsort(-bm25.get_scores(tokenize(query)), kind="stable")[:k]]

    report = {}
    for label, search in (("bm25", bm25_search), ("dense", dense_search)):
        hits, reciprocal_ranks, ranks = 0, [], []
        for query, target in zip(queries, targets):
            results = search(query)
            rank = results.index(target) + 1 if target in results else None
            ranks.append(rank)
            if rank is None:
                reciprocal_ranks.append(0.0)
            else:
                reciprocal_ranks.append(1.0 / rank)
                if rank <= 3:
                    hits += 1
        report[label + "_hit@3"] = round(hits / len(queries), 3)
        report[label + "_mrr"] = round(float(np.mean(reciprocal_ranks)), 3)
        report[label + "_ranks"] = ranks
    return report
```

---

### 4. Test Cases

**in1.txt**

```
1
12
Guardrail Overview. Project Guardrail hardens the company LLM endpoints against prompt injection and jailbreak attacks. It covers input sanitization, output filtering, and periodic red teaming.
Sentinel Overview. Sentinel provides continuous LLM security monitoring of production model traffic, detecting adversarial prompts and potential data exfiltration in real time.
Atlas Overview. Atlas is the unified data platform providing feature storage, batch pipelines, and governed access to analytics tables across the company.
Nimbus Overview. Nimbus manages the Kubernetes based infrastructure, autoscaling, and service mesh that host internal applications.
INC-2847 Postmortem. Incident INC-2847 involved a prompt injection attack against the Guardrail-protected chat endpoint. Root cause: output filter regression after the v2.3 deploy.
INC-3102 Postmortem. Incident INC-3102 was a Kubernetes autoscaler misconfiguration in Nimbus that caused cascading pod evictions during peak traffic.
RFC-014 Cross-Encoder Reranking. This RFC proposes adopting cross-encoder reranking on top of hybrid retrieval in the internal assistant. Recommended shortlist size: 20.
Data Retention Policy. Personal data is retained for at most 90 days in operational stores. Aggregated analytics data may be retained for 24 months.
How Guardrail Handles Jailbreaks. The defense pipeline layers instruction hierarchy tagging, adversarial suffix detection, and periodic red-team evaluations against a curated attack set.
Atlas Feature Store Internals. Feature values in Atlas are versioned by ingestion timestamp. Point-in-time joins use a snapshot table to prevent training-serving skew.
Nimbus Service Mesh Design. Nimbus uses a sidecar service mesh for mTLS and traffic shaping. Circuit breakers open on 5xx rates exceeding 5 percent for 30 seconds.
New Hire Onboarding Guide. New engineers are paired with a buddy for the first two weeks, complete the security training module, and ship a small starter task by end of week two.
4
how long do we hold on to the customer records before they get deleted
how often do we run the periodic red teaming exercises
what covers the security training for the company
what happened when the autoscaler evicted the pods during peak traffic
4
7
0
11
5
5
```

**out1.txt**

```
{'bm25_hit@3': 0.75, 'bm25_mrr': 0.75, 'bm25_ranks': [None, 1, 1, 1], 'dense_hit@3': 0.75, 'dense_mrr': 0.583, 'dense_ranks': [1, None, 3, 1]}
```

**in2.txt**

```
1
12
Guardrail Overview. Project Guardrail hardens the company LLM endpoints against prompt injection and jailbreak attacks. It covers input sanitization, output filtering, and periodic red teaming.
Sentinel Overview. Sentinel provides continuous LLM security monitoring of production model traffic, detecting adversarial prompts and potential data exfiltration in real time.
Atlas Overview. Atlas is the unified data platform providing feature storage, batch pipelines, and governed access to analytics tables across the company.
Nimbus Overview. Nimbus manages the Kubernetes based infrastructure, autoscaling, and service mesh that host internal applications.
INC-2847 Postmortem. Incident INC-2847 involved a prompt injection attack against the Guardrail-protected chat endpoint. Root cause: output filter regression after the v2.3 deploy.
INC-3102 Postmortem. Incident INC-3102 was a Kubernetes autoscaler misconfiguration in Nimbus that caused cascading pod evictions during peak traffic.
RFC-014 Cross-Encoder Reranking. This RFC proposes adopting cross-encoder reranking on top of hybrid retrieval in the internal assistant. Recommended shortlist size: 20.
Data Retention Policy. Personal data is retained for at most 90 days in operational stores. Aggregated analytics data may be retained for 24 months.
How Guardrail Handles Jailbreaks. The defense pipeline layers instruction hierarchy tagging, adversarial suffix detection, and periodic red-team evaluations against a curated attack set.
Atlas Feature Store Internals. Feature values in Atlas are versioned by ingestion timestamp. Point-in-time joins use a snapshot table to prevent training-serving skew.
Nimbus Service Mesh Design. Nimbus uses a sidecar service mesh for mTLS and traffic shaping. Circuit breakers open on 5xx rates exceeding 5 percent for 30 seconds.
New Hire Onboarding Guide. New engineers are paired with a buddy for the first two weeks, complete the security training module, and ship a small starter task by end of week two.
3
the security training module
the input sanitization and jailbreak attacks
the cascading pod evictions during peak traffic
3
11
0
5
3
```

**out2.txt**

```
{'bm25_hit@3': 1.0, 'bm25_mrr': 1.0, 'bm25_ranks': [1, 1, 1], 'dense_hit@3': 0.667, 'dense_mrr': 0.5, 'dense_ranks': [None, 2, 1]}
```

In [19]:
import re

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

EMBED_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def tokenize(text: str) -> list:
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())


def compare_retrievers(doc_texts: list, queries: list, targets: list, k: int) -> dict:
    matrix = np.asarray(EMBED_MODEL.encode(doc_texts, normalize_embeddings=True), dtype="float32")
    bm25 = BM25Okapi([tokenize(text) for text in doc_texts])

    def dense_search(query):
        query_vec = np.asarray(EMBED_MODEL.encode([query], normalize_embeddings=True), dtype="float32")[0]
        return [int(i) for i in np.argsort(-(matrix @ query_vec), kind="stable")[:k]]

    def bm25_search(query):
        return [int(i) for i in np.argsort(-bm25.get_scores(tokenize(query)), kind="stable")[:k]]

    report = {}
    for label, search in (("bm25", bm25_search), ("dense", dense_search)):
        hits, reciprocal_ranks, ranks = 0, [], []
        for query, target in zip(queries, targets):
            results = search(query)
            rank = results.index(target) + 1 if target in results else None
            ranks.append(rank)
            if rank is None:
                reciprocal_ranks.append(0.0)
            else:
                reciprocal_ranks.append(1.0 / rank)
                if rank <= 3:
                    hits += 1
        report[label + "_hit@3"] = round(hits / len(queries), 3)
        report[label + "_mrr"] = round(float(np.mean(reciprocal_ranks)), 3)
        report[label + "_ranks"] = ranks
    return report

In [20]:
doc_texts = [
    "Guardrail Overview. Project Guardrail hardens the company LLM endpoints against prompt injection and jailbreak attacks. It covers input sanitization, output filtering, and periodic red teaming.",
    "Sentinel Overview. Sentinel provides continuous LLM security monitoring of production model traffic, detecting adversarial prompts and potential data exfiltration in real time.",
    "Atlas Overview. Atlas is the unified data platform providing feature storage, batch pipelines, and governed access to analytics tables across the company.",
    "Nimbus Overview. Nimbus manages the Kubernetes based infrastructure, autoscaling, and service mesh that host internal applications.",
    "INC-2847 Postmortem. Incident INC-2847 involved a prompt injection attack against the Guardrail-protected chat endpoint. Root cause: output filter regression after the v2.3 deploy.",
    "INC-3102 Postmortem. Incident INC-3102 was a Kubernetes autoscaler misconfiguration in Nimbus that caused cascading pod evictions during peak traffic.",
    "RFC-014 Cross-Encoder Reranking. This RFC proposes adopting cross-encoder reranking on top of hybrid retrieval in the internal assistant. Recommended shortlist size: 20.",
    "Data Retention Policy. Personal data is retained for at most 90 days in operational stores. Aggregated analytics data may be retained for 24 months.",
    "How Guardrail Handles Jailbreaks. The defense pipeline layers instruction hierarchy tagging, adversarial suffix detection, and periodic red-team evaluations against a curated attack set.",
    "Atlas Feature Store Internals. Feature values in Atlas are versioned by ingestion timestamp. Point-in-time joins use a snapshot table to prevent training-serving skew.",
    "Nimbus Service Mesh Design. Nimbus uses a sidecar service mesh for mTLS and traffic shaping. Circuit breakers open on 5xx rates exceeding 5 percent for 30 seconds.",
    "New Hire Onboarding Guide. New engineers are paired with a buddy for the first two weeks, complete the security training module, and ship a small starter task by end of week two."
]
queries = [
    "how long do we hold on to the customer records before they get deleted",
    "how often do we run the periodic red teaming exercises",
    "what covers the security training for the company",
    "what happened when the autoscaler evicted the pods during peak traffic"
]
targets = [7, 0, 11, 5]
k = 5

compare_retrievers(doc_texts, queries, targets, k)

{'bm25_hit@3': 0.75,
 'bm25_mrr': 0.75,
 'bm25_ranks': [None, 1, 1, 1],
 'dense_hit@3': 0.75,
 'dense_mrr': 0.583,
 'dense_ranks': [1, None, 3, 1]}

In [21]:
doc_texts = [
    "Guardrail Overview. Project Guardrail hardens the company LLM endpoints against prompt injection and jailbreak attacks. It covers input sanitization, output filtering, and periodic red teaming.",
    "Sentinel Overview. Sentinel provides continuous LLM security monitoring of production model traffic, detecting adversarial prompts and potential data exfiltration in real time.",
    "Atlas Overview. Atlas is the unified data platform providing feature storage, batch pipelines, and governed access to analytics tables across the company.",
    "Nimbus Overview. Nimbus manages the Kubernetes based infrastructure, autoscaling, and service mesh that host internal applications.",
    "INC-2847 Postmortem. Incident INC-2847 involved a prompt injection attack against the Guardrail-protected chat endpoint. Root cause: output filter regression after the v2.3 deploy.",
    "INC-3102 Postmortem. Incident INC-3102 was a Kubernetes autoscaler misconfiguration in Nimbus that caused cascading pod evictions during peak traffic.",
    "RFC-014 Cross-Encoder Reranking. This RFC proposes adopting cross-encoder reranking on top of hybrid retrieval in the internal assistant. Recommended shortlist size: 20.",
    "Data Retention Policy. Personal data is retained for at most 90 days in operational stores. Aggregated analytics data may be retained for 24 months.",
    "How Guardrail Handles Jailbreaks. The defense pipeline layers instruction hierarchy tagging, adversarial suffix detection, and periodic red-team evaluations against a curated attack set.",
    "Atlas Feature Store Internals. Feature values in Atlas are versioned by ingestion timestamp. Point-in-time joins use a snapshot table to prevent training-serving skew.",
    "Nimbus Service Mesh Design. Nimbus uses a sidecar service mesh for mTLS and traffic shaping. Circuit breakers open on 5xx rates exceeding 5 percent for 30 seconds.",
    "New Hire Onboarding Guide. New engineers are paired with a buddy for the first two weeks, complete the security training module, and ship a small starter task by end of week two."
]
queries = [
    "the security training module",
    "the input sanitization and jailbreak attacks",
    "the cascading pod evictions during peak traffic"
]
targets = [11, 0, 5]
k = 3

compare_retrievers(doc_texts, queries, targets, k)

{'bm25_hit@3': 1.0,
 'bm25_mrr': 1.0,
 'bm25_ranks': [1, 1, 1],
 'dense_hit@3': 0.667,
 'dense_mrr': 0.5,
 'dense_ranks': [None, 2, 1]}

### [Class 13] Question 12. The baseline number everything else is measured against

**Context:** Before any optimizer is worth running there has to be a number to beat, and that number is
only as honest as the metric behind it. A model asked for one word does not reliably return one word: it
returns `Positive.`, or `POSITIVE`, or a whole sentence with the label buried in it. A metric that
compares strings exactly scores all of those wrong and reports a baseline far below the truth, which then
makes any optimizer look like a miracle. The metric has to normalize what the model actually emits.

**Problem Statement:**

Using DSPy, write a function `baseline_accuracy(headlines, gold, replies)` that evaluates a
single-shot classifier over a labelled set.

1. Declare a signature class `FinancialSentiment` at module level, with this exact docstring:

   ```
   Classify the sentiment of a financial news headline from the perspective of an equity investor.
   ```

   and exactly two fields: an input field `headline` described as
   `"a short financial news headline"`, and an output field `sentiment` described as
   `"exactly one word: positive, negative, or neutral"`.
2. Configure DSPy with a stand-in language model that replays a fixed reply per headline, so the run is
   reproducible: `DummyLM(answers)`, where `answers` maps each headline to `{"sentiment": reply}` using
   the reply at the same position in `replies`.
3. Write a metric `sentiment_match(example, pred, trace=None)` returning `1.0` or `0.0`. Strip and lowercase both the
   example's gold label and the prediction's `sentiment`. Then, if the lowercased prediction contains
   any of `positive`, `negative`, `neutral`, replace it with the first of those three — checked in that
   order.
4. Build the evaluation set as DSPy examples carrying `headline` and `sentiment`, declaring `headline` as
   the input field, and evaluate `dspy.Predict(FinancialSentiment)` over it with `dspy.evaluate.Evaluate`
   configured with that metric, `num_threads=1`, no progress display and no result table.
5. Return a dictionary with exactly four keys: `"score"`, the evaluation's score as reported;
   `"per_example"`, the metric value for each example in devset order; `"wrong"`, the headlines whose
   metric value was below `1.0`, in devset order; and `"parsed"`, the `sentiment` carried by the
   prediction for each example, in devset order, exactly as the prediction reports it.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: headlines (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 2: gold (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 3: replies (list_string)
    Line 1: N (list length)
    Next N lines: one string each
```

**Output Format:**

For each test case, print the result of `baseline_accuracy()` on a new line.

**Sample Input:**

```
headlines = [
    "Retailer reports a sharp drop in same-store sales",
    "Board confirms the annual dividend at last year's level",
    "Bank raises full-year profit outlook after strong lending"
]
gold = ["negative", "neutral", "positive"]
replies = ["negative", "  neutral  ", "the sentiment is positive"]
```

**Sample Output:**

```
{'score': 100.0, 'per_example': [1.0, 1.0, 1.0], 'wrong': [], 'parsed': ['negative', 'neutral', 'the sentiment is positive']}
```

**Note:** The stand-in model is imported from `dspy.utils.dummies`. The score is whatever the evaluator
reports — a percentage, not a fraction — so read it off the evaluation result rather than recomputing it.
The evaluator also exposes its per-example results as
`.results`, a list of `(Example, Prediction, float)` triples in devset order; take both
`"per_example"` and `"parsed"` from there. The reply the stand-in model was handed is not necessarily
what reaches the prediction, so `"parsed"` has to be read off the `Prediction` and never off `replies`.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: baseline_accuracy
RETURN TYPE: Dictionary

ARGUMENTS:
1. headlines → List of Strings
2. gold      → List of Strings
3. replies   → List of Strings
```

---

### 3. Reference Solution Code

```python
import dspy
from dspy.evaluate import Evaluate
from dspy.utils.dummies import DummyLM


class FinancialSentiment(dspy.Signature):
    """Classify the sentiment of a financial news headline from the perspective of an equity investor."""

    headline: str = dspy.InputField(desc="a short financial news headline")
    sentiment: str = dspy.OutputField(desc="exactly one word: positive, negative, or neutral")


def sentiment_match(example, pred, trace=None) -> float:
    gold = example.sentiment.strip().lower()
    predicted = (pred.sentiment or "").strip().lower()
    for label in ["positive", "negative", "neutral"]:
        if label in predicted:
            predicted = label
            break
    return float(gold == predicted)


def baseline_accuracy(headlines: list, gold: list, replies: list) -> dict:
    dspy.configure(lm=DummyLM({h: {"sentiment": r} for h, r in zip(headlines, replies)}))

    devset = [dspy.Example(headline=h, sentiment=g).with_inputs("headline")
              for h, g in zip(headlines, gold)]

    evaluator = Evaluate(devset=devset, metric=sentiment_match, num_threads=1,
                         display_progress=False, display_table=0)
    result = evaluator(dspy.Predict(FinancialSentiment))

    return {"score": result.score,
            "per_example": [score for _, _, score in result.results],
            "wrong": [h for h, (_, _, score) in zip(headlines, result.results) if score < 1.0],
            "parsed": [pred.sentiment for _, pred, _ in result.results]}
```

---

### 4. Test Cases

**in1.txt**

```
1
4
Nokia's third-quarter profit fell short of analyst expectations
Company beats Q3 earnings estimates by 15 percent
The board will meet on Tuesday to review the agenda
Regulator opens investigation into accounting practices
4
negative
positive
neutral
negative
4
[[ ## sentiment ## ]] negative
Positive.
the sentiment is neutral
positive
```

**out1.txt**

```
{'score': 50.0, 'per_example': [0.0, 1.0, 1.0, 0.0], 'wrong': ["Nokia's third-quarter profit fell short of analyst expectations", 'Regulator opens investigation into accounting practices'], 'parsed': ['', 'Positive.', 'the sentiment is neutral', 'positive']}
```

**in2.txt**

```
1
3
Shares jumped 12 percent after the buyback announcement
Dividend left unchanged for the fourth straight quarter
Supplier warns of a shortfall in component deliveries
3
positive
neutral
negative
3
[[ ## sentiment ## ]] POSITIVE
  neutral  
Based on the report the sentiment is negative for investors
```

**out2.txt**

```
{'score': 66.67, 'per_example': [0.0, 1.0, 1.0], 'wrong': ['Shares jumped 12 percent after the buyback announcement'], 'parsed': ['', 'neutral', 'Based on the report the sentiment is negative for investors']}
```

**in3.txt**

```
1
3
Airline lifts full-year guidance after record summer bookings
Chief financial officer resigns with immediate effect
Annual general meeting scheduled for the second week of May
3
positive
negative
neutral
3
[[ ## sentiment ## ]] positive
neutral [[ ## completed ## ]] though the abrupt exit is negative for confidence
  Neutral.  
```

**out3.txt**

```
{'score': 66.67, 'per_example': [0.0, 1.0, 1.0], 'wrong': ['Airline lifts full-year guidance after record summer bookings'], 'parsed': ['', 'neutral [[ ## completed ## ]] though the abrupt exit is negative for confidence', 'Neutral.']}
```


In [4]:
import dspy
from dspy.evaluate import Evaluate
from dspy.utils.dummies import DummyLM


class FinancialSentiment(dspy.Signature):
    """Classify the sentiment of a financial news headline from the perspective of an equity investor."""

    headline: str = dspy.InputField(desc="a short financial news headline")
    sentiment: str = dspy.OutputField(desc="exactly one word: positive, negative, or neutral")


def sentiment_match(example, pred, trace=None) -> float:
    gold = example.sentiment.strip().lower()
    predicted = (pred.sentiment or "").strip().lower()
    for label in ["positive", "negative", "neutral"]:
        if label in predicted:
            predicted = label
            break
    return float(gold == predicted)


def baseline_accuracy(headlines: list, gold: list, replies: list) -> dict:
    dspy.configure(lm=DummyLM({h: {"sentiment": r} for h, r in zip(headlines, replies)}))

    devset = [dspy.Example(headline=h, sentiment=g).with_inputs("headline")
              for h, g in zip(headlines, gold)]

    evaluator = Evaluate(devset=devset, metric=sentiment_match, num_threads=1,
                         display_progress=False, display_table=0)
    result = evaluator(dspy.Predict(FinancialSentiment))

    return {"score": result.score,
            "per_example": [score for _, _, score in result.results],
            "wrong": [h for h, (_, _, score) in zip(headlines, result.results) if score < 1.0],
            "parsed": [pred.sentiment for _, pred, _ in result.results]}


In [1]:
headlines = [
    "Nokia's third-quarter profit fell short of analyst expectations",
    "Company beats Q3 earnings estimates by 15 percent",
    "The board will meet on Tuesday to review the agenda",
    "Regulator opens investigation into accounting practices"
]
gold = ["negative", "positive", "neutral", "negative"]
replies = ["[[ ## sentiment ## ]] negative", "Positive.", "the sentiment is neutral", "positive"]

baseline_accuracy(headlines, gold, replies)


{'score': 50.0, 'per_example': [0.0, 1.0, 1.0, 0.0], 'wrong': ["Nokia's third-quarter profit fell short of analyst expectations", 'Regulator opens investigation into accounting practices'], 'parsed': ['', 'Positive.', 'the sentiment is neutral', 'positive']}


In [1]:
headlines = [
    "Shares jumped 12 percent after the buyback announcement",
    "Dividend left unchanged for the fourth straight quarter",
    "Supplier warns of a shortfall in component deliveries"
]
gold = ["positive", "neutral", "negative"]
replies = ["[[ ## sentiment ## ]] POSITIVE", "  neutral  ", "Based on the report the sentiment is negative for investors"]

baseline_accuracy(headlines, gold, replies)


{'score': 66.67, 'per_example': [0.0, 1.0, 1.0], 'wrong': ['Shares jumped 12 percent after the buyback announcement'], 'parsed': ['', 'neutral', 'Based on the report the sentiment is negative for investors']}


In [1]:
headlines = [
    "Airline lifts full-year guidance after record summer bookings",
    "Chief financial officer resigns with immediate effect",
    "Annual general meeting scheduled for the second week of May"
]
gold = ["positive", "negative", "neutral"]
replies = [
    "[[ ## sentiment ## ]] positive",
    "neutral [[ ## completed ## ]] though the abrupt exit is negative for confidence",
    "  Neutral.  "
]

baseline_accuracy(headlines, gold, replies)


{'score': 66.67, 'per_example': [0.0, 1.0, 1.0], 'wrong': ['Airline lifts full-year guidance after record summer bookings'], 'parsed': ['', 'neutral [[ ## completed ## ]] though the abrupt exit is negative for confidence', 'Neutral.']}


### [Class 13] Question 13. The optimizer edits the prompt, not you

**Context:** Bootstrapping few-shot demonstrations is the cheapest optimizer there is, and it is worth
understanding exactly because it is so simple. It runs the program over the training set, keeps the
traces where the metric said the answer was right, and stops as soon as it has collected as many as it
was allowed. Nothing hand-written enters the prompt: the demonstrations are the program's own successful
runs, and they arrive in the compiled prompt as extra conversational turns, which is the whole reason the
compiled prompt is larger than the one you started with.

**Problem Statement:**

Using DSPy, write a function
`compile_demos(headlines, gold, replies, max_demos, probe_headline)` that compiles a classifier and
reports what the optimizer put in the prompt.

1. Declare a signature class `FinancialSentiment` at module level, with this exact docstring:

   ```
   Classify the sentiment of a financial news headline from the perspective of an equity investor.
   ```

   and exactly two fields: an input field `headline` described as
   `"a short financial news headline"`, and an output field `sentiment` described as
   `"exactly one word: positive, negative, or neutral"`.
2. Configure DSPy with a stand-in language model that replays a fixed reply per headline, so the run is
   reproducible: `DummyLM(answers)`, where `answers` maps each training headline to
   `{"reasoning": "Assessed from the headline.", "sentiment": reply}` using the reply at the same
   position in `replies`, plus one more entry for `probe_headline`, with those same two keys, whose
   `sentiment` is `"neutral"`.
3. Write a metric `sentiment_match(example, pred, trace=None)` returning `1.0` or `0.0`. It compares the
   example's gold label, stripped and lowercased, against the prediction's `sentiment`, stripped and
   lowercased — but first, if any of `positive`, `negative`, `neutral` occurs anywhere in that
   prediction, the prediction is replaced by the first such label found when they are checked in that
   order.
4. Build the training set as DSPy examples carrying `headline` and `sentiment`, declaring `headline` as
   the input field. Compile a `dspy.ChainOfThought(FinancialSentiment)` student with `BootstrapFewShot`
   using that metric, at most `max_demos` bootstrapped demonstrations, **no** raw labelled
   demonstrations, and a single round.
5. Call the compiled program once on `probe_headline`, then read the messages of that call off the
   language model's history.
6. Return a dictionary with exactly five keys: `"n_demos"`, how many demonstrations the compiled program
   ended up with; `"demo_headlines"`, the `headline` of each demonstration in order; `"n_messages"`, how
   many messages that last call sent; `"roles"`, the role of each of those messages in order; and
   `"first_demo_turn"`, the `content` of the first message of that call whose role is `assistant` —
   DSPy's rendering of one bootstrapped demonstration.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: headlines (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 2: gold (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 3: replies (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 4: max_demos (integer)
    Single line: one integer
  Argument 5: probe_headline (string)
    Single line: a string
```

**Output Format:**

For each test case, print the result of `compile_demos()` on a new line.

**Sample Input:**

```
headlines = [
    "Retailer reports a sharp drop in same-store sales",
    "Bank raises full-year profit outlook after strong lending",
    "Annual general meeting scheduled for the second week of May",
    "Insurer books a large reserve charge for storm claims"
]
gold = ["negative", "positive", "neutral", "negative"]
replies = ["negative", "positive", "neutral", "negative"]
max_demos = 3
probe_headline = "Supplier warns of a shortfall in component deliveries"
```

**Sample Output:**

```
Bootstrapped 3 full traces after 3 examples for up to 1 rounds, amounting to 3 attempts.
{'n_demos': 3, 'demo_headlines': ['Retailer reports a sharp drop in same-store sales', 'Bank raises full-year profit outlook after strong lending', 'Annual general meeting scheduled for the second week of May'], 'n_messages': 8, 'roles': ['system', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user'], 'first_demo_turn': '[[ ## reasoning ## ]]\nAssessed from the headline.\n\n[[ ## sentiment ## ]]\nnegative\n\n[[ ## completed ## ]]\n'}
```

**Note:** The stand-in model is imported from `dspy.utils.dummies`. The messages of a call are at
`lm.history[-1]["messages"]`, and the compiled program's demonstrations sit on its predictor as
`.demos`; a demonstration behaves like an example — its fields are read by name. Do not build the
message list or the demo turn yourself: report the `content` string the compiled program actually sent,
character for character, whitespace included. The optimizer prints a one-line `Bootstrapped ...` summary to stdout as it compiles. Leave it where it lands — it is part of the output, as the sample shows.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: compile_demos
RETURN TYPE: Dictionary

ARGUMENTS:
1. headlines      → List of Strings
2. gold           → List of Strings
3. replies        → List of Strings
4. max_demos      → Integer
5. probe_headline → String
```

---

### 3. Reference Solution Code

```python
import dspy
from dspy.teleprompt import BootstrapFewShot
from dspy.utils.dummies import DummyLM


class FinancialSentiment(dspy.Signature):
    """Classify the sentiment of a financial news headline from the perspective of an equity investor."""

    headline: str = dspy.InputField(desc="a short financial news headline")
    sentiment: str = dspy.OutputField(desc="exactly one word: positive, negative, or neutral")


def sentiment_match(example, pred, trace=None) -> float:
    gold = example.sentiment.strip().lower()
    predicted = (pred.sentiment or "").strip().lower()
    for label in ["positive", "negative", "neutral"]:
        if label in predicted:
            predicted = label
            break
    return float(gold == predicted)


def compile_demos(headlines: list, gold: list, replies: list,
                  max_demos: int, probe_headline: str) -> dict:
    answers = {h: {"reasoning": "Assessed from the headline.", "sentiment": r}
               for h, r in zip(headlines, replies)}
    answers[probe_headline] = {"reasoning": "Assessed from the headline.", "sentiment": "neutral"}
    lm = DummyLM(answers)
    dspy.configure(lm=lm)

    trainset = [dspy.Example(headline=h, sentiment=g).with_inputs("headline")
                for h, g in zip(headlines, gold)]

    optimizer = BootstrapFewShot(metric=sentiment_match, max_bootstrapped_demos=max_demos,
                                 max_labeled_demos=0, max_rounds=1)
    compiled = optimizer.compile(student=dspy.ChainOfThought(FinancialSentiment), trainset=trainset)

    compiled(headline=probe_headline)
    messages = lm.history[-1]["messages"]

    return {"n_demos": len(compiled.predict.demos),
            "demo_headlines": [demo["headline"] for demo in compiled.predict.demos],
            "n_messages": len(messages),
            "roles": [message["role"] for message in messages],
            "first_demo_turn": next(message["content"] for message in messages
                                    if message["role"] == "assistant")}
```

---

### 4. Test Cases

**in1.txt**

```
1
5
Nokia's third-quarter profit fell short of analyst expectations
Company beats Q3 earnings estimates by 15 percent
The board will meet on Tuesday to review the agenda
Regulator opens investigation into accounting practices
Shares jumped 12 percent after the buyback announcement
5
negative
positive
neutral
negative
positive
5
[[ ## sentiment ## ]] negative
Positive.
positive
negative
positive
4
Supplier warns of a shortfall in component deliveries
```

**out1.txt**

```
Bootstrapped 3 full traces after 4 examples for up to 1 rounds, amounting to 5 attempts.
{'n_demos': 3, 'demo_headlines': ['Company beats Q3 earnings estimates by 15 percent', 'Regulator opens investigation into accounting practices', 'Shares jumped 12 percent after the buyback announcement'], 'n_messages': 8, 'roles': ['system', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user'], 'first_demo_turn': '[[ ## reasoning ## ]]\nAssessed from the headline.\n\n[[ ## sentiment ## ]]\nPositive.\n\n[[ ## completed ## ]]\n'}
```

**in2.txt**

```
1
5
Nokia's third-quarter profit fell short of analyst expectations
Company beats Q3 earnings estimates by 15 percent
The board will meet on Tuesday to review the agenda
Regulator opens investigation into accounting practices
Shares jumped 12 percent after the buyback announcement
5
negative
positive
neutral
negative
positive
5
[[ ## sentiment ## ]] negative
positive [[ ## completed ## ]] guidance raised
positive
negative
positive
2
Supplier warns of a shortfall in component deliveries
```

**out2.txt**

```
Bootstrapped 2 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.
{'n_demos': 2, 'demo_headlines': ['Company beats Q3 earnings estimates by 15 percent', 'Regulator opens investigation into accounting practices'], 'n_messages': 6, 'roles': ['system', 'user', 'assistant', 'user', 'assistant', 'user'], 'first_demo_turn': '[[ ## reasoning ## ]]\nAssessed from the headline.\n\n[[ ## sentiment ## ]]\npositive [[ ## completed ## ]] guidance raised\n\n[[ ## completed ## ]]\n'}
```


In [7]:
import dspy
from dspy.teleprompt import BootstrapFewShot
from dspy.utils.dummies import DummyLM


class FinancialSentiment(dspy.Signature):
    """Classify the sentiment of a financial news headline from the perspective of an equity investor."""

    headline: str = dspy.InputField(desc="a short financial news headline")
    sentiment: str = dspy.OutputField(desc="exactly one word: positive, negative, or neutral")


def sentiment_match(example, pred, trace=None) -> float:
    gold = example.sentiment.strip().lower()
    predicted = (pred.sentiment or "").strip().lower()
    for label in ["positive", "negative", "neutral"]:
        if label in predicted:
            predicted = label
            break
    return float(gold == predicted)


def compile_demos(headlines: list, gold: list, replies: list,
                  max_demos: int, probe_headline: str) -> dict:
    answers = {h: {"reasoning": "Assessed from the headline.", "sentiment": r}
               for h, r in zip(headlines, replies)}
    answers[probe_headline] = {"reasoning": "Assessed from the headline.", "sentiment": "neutral"}
    lm = DummyLM(answers)
    dspy.configure(lm=lm)

    trainset = [dspy.Example(headline=h, sentiment=g).with_inputs("headline")
                for h, g in zip(headlines, gold)]

    optimizer = BootstrapFewShot(metric=sentiment_match, max_bootstrapped_demos=max_demos,
                                 max_labeled_demos=0, max_rounds=1)
    compiled = optimizer.compile(student=dspy.ChainOfThought(FinancialSentiment), trainset=trainset)

    compiled(headline=probe_headline)
    messages = lm.history[-1]["messages"]

    return {"n_demos": len(compiled.predict.demos),
            "demo_headlines": [demo["headline"] for demo in compiled.predict.demos],
            "n_messages": len(messages),
            "roles": [message["role"] for message in messages],
            "first_demo_turn": next(message["content"] for message in messages
                                    if message["role"] == "assistant")}


In [1]:
headlines = [
    "Nokia's third-quarter profit fell short of analyst expectations",
    "Company beats Q3 earnings estimates by 15 percent",
    "The board will meet on Tuesday to review the agenda",
    "Regulator opens investigation into accounting practices",
    "Shares jumped 12 percent after the buyback announcement"
]
gold = ["negative", "positive", "neutral", "negative", "positive"]
replies = ["[[ ## sentiment ## ]] negative", "Positive.", "positive", "negative", "positive"]
max_demos = 4
probe_headline = "Supplier warns of a shortfall in component deliveries"

compile_demos(headlines, gold, replies, max_demos, probe_headline)


Bootstrapped 3 full traces after 4 examples for up to 1 rounds, amounting to 5 attempts.
{'n_demos': 3, 'demo_headlines': ['Company beats Q3 earnings estimates by 15 percent', 'Regulator opens investigation into accounting practices', 'Shares jumped 12 percent after the buyback announcement'], 'n_messages': 8, 'roles': ['system', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user'], 'first_demo_turn': '[[ ## reasoning ## ]]\nAssessed from the headline.\n\n[[ ## sentiment ## ]]\nPositive.\n\n[[ ## completed ## ]]\n'}


In [1]:
headlines = [
    "Nokia's third-quarter profit fell short of analyst expectations",
    "Company beats Q3 earnings estimates by 15 percent",
    "The board will meet on Tuesday to review the agenda",
    "Regulator opens investigation into accounting practices",
    "Shares jumped 12 percent after the buyback announcement"
]
gold = ["negative", "positive", "neutral", "negative", "positive"]
replies = [
    "[[ ## sentiment ## ]] negative",
    "positive [[ ## completed ## ]] guidance raised",
    "positive",
    "negative",
    "positive"
]
max_demos = 2
probe_headline = "Supplier warns of a shortfall in component deliveries"

compile_demos(headlines, gold, replies, max_demos, probe_headline)


Bootstrapped 2 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.
{'n_demos': 2, 'demo_headlines': ['Company beats Q3 earnings estimates by 15 percent', 'Regulator opens investigation into accounting practices'], 'n_messages': 6, 'roles': ['system', 'user', 'assistant', 'user', 'assistant', 'user'], 'first_demo_turn': '[[ ## reasoning ## ]]\nAssessed from the headline.\n\n[[ ## sentiment ## ]]\npositive [[ ## completed ## ]] guidance raised\n\n[[ ## completed ## ]]\n'}


### [Class 13] Question 14. Shipping the compiled program

**Context:** A compiled program is not a model — it is a small artifact describing how to prompt one, and
that artifact is what gets committed, reviewed and deployed. Two things follow. It has to be inspectable,
because the demonstrations inside it were chosen by a machine and somebody has to be able to read what
went in. And reloading it into a fresh, unoptimized module has to reproduce the optimized behaviour
exactly, or the deployed service is not running what was tested.

**Problem Statement:**

Using DSPy, write a function `ship_program(headlines, gold, replies, max_demos, probe_headline)` that
compiles a classifier, saves it, inspects the artifact, and reloads it.

1. Declare a signature class `FinancialSentiment` at module level, with this exact docstring:

   ```
   Classify the sentiment of a financial news headline from the perspective of an equity investor.
   ```

   and exactly two fields: an input field `headline` described as
   `"a short financial news headline"`, and an output field `sentiment` described as
   `"exactly one word: positive, negative, or neutral"`.
2. Configure DSPy with a stand-in language model that replays a fixed reply per headline, so the run is
   reproducible: `DummyLM(answers)`, where `answers` maps each training headline to
   `{"reasoning": "Assessed from the headline.", "sentiment": reply}` using the reply at the same
   position in `replies`, plus one more entry for `probe_headline`, with those same two keys, whose
   `sentiment` is `"neutral"`.
3. Write a metric `sentiment_match(example, pred, trace=None)` returning `1.0` or `0.0`. It compares the
   example's gold label, stripped and lowercased, against the prediction's `sentiment`, stripped and
   lowercased — but first, if any of `positive`, `negative`, `neutral` occurs anywhere in that
   prediction, the prediction is replaced by the first such label found when they are checked in that
   order.
4. Build the training set as DSPy examples carrying `headline` and `sentiment`, declaring `headline` as
   the input field, and compile a `dspy.ChainOfThought(FinancialSentiment)` student with
   `BootstrapFewShot` using that metric, at most `max_demos` bootstrapped demonstrations, **no** raw
   labelled demonstrations, and a single round.
5. Call the compiled program once on `probe_headline` and keep the messages of that call from the
   language model's history.
6. Save the compiled program with the module's own save-to-JSON call, then read that file back as
   plain JSON.
7. Load the saved file into a **fresh** `dspy.ChainOfThought(FinancialSentiment)` and call it once on
   `probe_headline`, then take the messages of that call.
8. Return a dictionary with exactly four keys: `"artifact_keys"`, the artifact's top-level keys sorted
   ascending; `"n_saved_demos"`, how many demonstrations it recorded; `"saved_demo_headlines"`, the
   `headline` recorded in each saved demonstration, in the order the artifact stores them; and
   `"prompt_unchanged"`, whether the reloaded program sent exactly the same messages as the compiled
   one.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: headlines (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 2: gold (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 3: replies (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 4: max_demos (integer)
    Single line: one integer
  Argument 5: probe_headline (string)
    Single line: a string
```

**Output Format:**

For each test case, print the result of `ship_program()` on a new line.

**Sample Input:**

```
headlines = [
    "Retailer reports a sharp drop in same-store sales",
    "Bank raises full-year profit outlook after strong lending",
    "Annual general meeting scheduled for the second week of May"
]
gold = ["negative", "positive", "neutral"]
replies = ["negative", "positive", "neutral"]
max_demos = 2
probe_headline = "Supplier warns of a shortfall in component deliveries"
```

**Sample Output:**

```
Bootstrapped 2 full traces after 2 examples for up to 1 rounds, amounting to 2 attempts.
{'artifact_keys': ['metadata', 'predict'], 'n_saved_demos': 2, 'saved_demo_headlines': ['Retailer reports a sharp drop in same-store sales', 'Bank raises full-year profit outlook after strong lending'], 'prompt_unchanged': True}
```

**Note:** The stand-in model is imported from `dspy.utils.dummies`. The saved demonstrations sit in the
`"demos"` list under the artifact's `"predict"` entry, and the messages of a call are at
`lm.history[-1]["messages"]`. The artifact's shape is DSPy's, not yours — report what it wrote rather
than constructing it, and take the headlines from the file you read back rather than from `headlines`.
Write the file somewhere temporary; its path is not part of the answer. The optimizer prints a one-line `Bootstrapped ...` summary to stdout as it compiles. Leave it where it lands — it is part of the output, as the sample shows.

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: ship_program
RETURN TYPE: Dictionary

ARGUMENTS:
1. headlines      → List of Strings
2. gold           → List of Strings
3. replies        → List of Strings
4. max_demos      → Integer
5. probe_headline → String
```

---

### 3. Reference Solution Code

```python
import json
import os
import tempfile

import dspy
from dspy.teleprompt import BootstrapFewShot
from dspy.utils.dummies import DummyLM


class FinancialSentiment(dspy.Signature):
    """Classify the sentiment of a financial news headline from the perspective of an equity investor."""

    headline: str = dspy.InputField(desc="a short financial news headline")
    sentiment: str = dspy.OutputField(desc="exactly one word: positive, negative, or neutral")


def sentiment_match(example, pred, trace=None) -> float:
    gold = example.sentiment.strip().lower()
    predicted = (pred.sentiment or "").strip().lower()
    for label in ["positive", "negative", "neutral"]:
        if label in predicted:
            predicted = label
            break
    return float(gold == predicted)


def ship_program(headlines: list, gold: list, replies: list,
                 max_demos: int, probe_headline: str) -> dict:
    answers = {h: {"reasoning": "Assessed from the headline.", "sentiment": r}
               for h, r in zip(headlines, replies)}
    answers[probe_headline] = {"reasoning": "Assessed from the headline.", "sentiment": "neutral"}
    lm = DummyLM(answers)
    dspy.configure(lm=lm)

    trainset = [dspy.Example(headline=h, sentiment=g).with_inputs("headline")
                for h, g in zip(headlines, gold)]
    compiled = BootstrapFewShot(metric=sentiment_match, max_bootstrapped_demos=max_demos,
                                max_labeled_demos=0, max_rounds=1).compile(
        student=dspy.ChainOfThought(FinancialSentiment), trainset=trainset)

    compiled(headline=probe_headline)
    before = lm.history[-1]["messages"]

    path = os.path.join(tempfile.mkdtemp(), "financial_sentiment_compiled.json")
    compiled.save(path)
    with open(path) as handle:
        artifact = json.load(handle)

    reloaded = dspy.ChainOfThought(FinancialSentiment)
    reloaded.load(path)
    reloaded(headline=probe_headline)
    after = lm.history[-1]["messages"]

    return {"artifact_keys": sorted(artifact.keys()),
            "n_saved_demos": len(artifact["predict"]["demos"]),
            "saved_demo_headlines": [demo["headline"] for demo in artifact["predict"]["demos"]],
            "prompt_unchanged": before == after}
```

---

### 4. Test Cases

**in1.txt**

```
1
4
Nokia's third-quarter profit fell short of analyst expectations
Company beats Q3 earnings estimates by 15 percent
The board will meet on Tuesday to review the agenda
Regulator opens investigation into accounting practices
4
negative
positive
neutral
negative
4
[[ ## sentiment ## ]] negative
Positive.
positive
negative
4
Supplier warns of a shortfall in component deliveries
```

**out1.txt**

```
Bootstrapped 2 full traces after 3 examples for up to 1 rounds, amounting to 4 attempts.
{'artifact_keys': ['metadata', 'predict'], 'n_saved_demos': 2, 'saved_demo_headlines': ['Company beats Q3 earnings estimates by 15 percent', 'Regulator opens investigation into accounting practices'], 'prompt_unchanged': True}
```

**in2.txt**

```
1
3
Dividend left unchanged for the fourth straight quarter
Shares jumped 12 percent after the buyback announcement
Auditor flags material weakness in internal controls
3
neutral
positive
negative
3
[[ ## sentiment ## ]] neutral
neutral [[ ## completed ## ]] though buybacks are read as positive by the market
Negative for the shares.
2
Supplier warns of a shortfall in component deliveries
```

**out2.txt**

```
Bootstrapped 2 full traces after 2 examples for up to 1 rounds, amounting to 3 attempts.
{'artifact_keys': ['metadata', 'predict'], 'n_saved_demos': 2, 'saved_demo_headlines': ['Shares jumped 12 percent after the buyback announcement', 'Auditor flags material weakness in internal controls'], 'prompt_unchanged': True}
```


In [10]:
import json
import os
import tempfile

import dspy
from dspy.teleprompt import BootstrapFewShot
from dspy.utils.dummies import DummyLM


class FinancialSentiment(dspy.Signature):
    """Classify the sentiment of a financial news headline from the perspective of an equity investor."""

    headline: str = dspy.InputField(desc="a short financial news headline")
    sentiment: str = dspy.OutputField(desc="exactly one word: positive, negative, or neutral")


def sentiment_match(example, pred, trace=None) -> float:
    gold = example.sentiment.strip().lower()
    predicted = (pred.sentiment or "").strip().lower()
    for label in ["positive", "negative", "neutral"]:
        if label in predicted:
            predicted = label
            break
    return float(gold == predicted)


def ship_program(headlines: list, gold: list, replies: list,
                 max_demos: int, probe_headline: str) -> dict:
    answers = {h: {"reasoning": "Assessed from the headline.", "sentiment": r}
               for h, r in zip(headlines, replies)}
    answers[probe_headline] = {"reasoning": "Assessed from the headline.", "sentiment": "neutral"}
    lm = DummyLM(answers)
    dspy.configure(lm=lm)

    trainset = [dspy.Example(headline=h, sentiment=g).with_inputs("headline")
                for h, g in zip(headlines, gold)]
    compiled = BootstrapFewShot(metric=sentiment_match, max_bootstrapped_demos=max_demos,
                                max_labeled_demos=0, max_rounds=1).compile(
        student=dspy.ChainOfThought(FinancialSentiment), trainset=trainset)

    compiled(headline=probe_headline)
    before = lm.history[-1]["messages"]

    path = os.path.join(tempfile.mkdtemp(), "financial_sentiment_compiled.json")
    compiled.save(path)
    with open(path) as handle:
        artifact = json.load(handle)

    reloaded = dspy.ChainOfThought(FinancialSentiment)
    reloaded.load(path)
    reloaded(headline=probe_headline)
    after = lm.history[-1]["messages"]

    return {"artifact_keys": sorted(artifact.keys()),
            "n_saved_demos": len(artifact["predict"]["demos"]),
            "saved_demo_headlines": [demo["headline"] for demo in artifact["predict"]["demos"]],
            "prompt_unchanged": before == after}


In [1]:
headlines = [
    "Nokia's third-quarter profit fell short of analyst expectations",
    "Company beats Q3 earnings estimates by 15 percent",
    "The board will meet on Tuesday to review the agenda",
    "Regulator opens investigation into accounting practices"
]
gold = ["negative", "positive", "neutral", "negative"]
replies = ["[[ ## sentiment ## ]] negative", "Positive.", "positive", "negative"]
max_demos = 4
probe_headline = "Supplier warns of a shortfall in component deliveries"

ship_program(headlines, gold, replies, max_demos, probe_headline)


Bootstrapped 2 full traces after 3 examples for up to 1 rounds, amounting to 4 attempts.
{'artifact_keys': ['metadata', 'predict'], 'n_saved_demos': 2, 'saved_demo_headlines': ['Company beats Q3 earnings estimates by 15 percent', 'Regulator opens investigation into accounting practices'], 'prompt_unchanged': True}


In [1]:
headlines = [
    "Dividend left unchanged for the fourth straight quarter",
    "Shares jumped 12 percent after the buyback announcement",
    "Auditor flags material weakness in internal controls"
]
gold = ["neutral", "positive", "negative"]
replies = [
    "[[ ## sentiment ## ]] neutral",
    "neutral [[ ## completed ## ]] though buybacks are read as positive by the market",
    "Negative for the shares."
]
max_demos = 2
probe_headline = "Supplier warns of a shortfall in component deliveries"

ship_program(headlines, gold, replies, max_demos, probe_headline)


Bootstrapped 2 full traces after 2 examples for up to 1 rounds, amounting to 3 attempts.
{'artifact_keys': ['metadata', 'predict'], 'n_saved_demos': 2, 'saved_demo_headlines': ['Shares jumped 12 percent after the buyback announcement', 'Auditor flags material weakness in internal controls'], 'prompt_unchanged': True}


### [Class 18] Question 15. How a U-Net Tells One Timestep From Another

**Context:** One U-Net handles every step of the reverse process, and it can only apply the right
amount of denoising if it knows where on the noise ladder it currently sits. The integer timestep
is therefore expanded into a sinusoidal **time embedding** and injected into the network's layers.
Whether two steps are distinguishable to the network is a property of that embedding, not of the
integers — so a sampler's step plan is worth probing before it is wired into a loop: embed the
steps it will actually visit and measure how alike consecutive ones look.

**Problem Statement:**

Using the `diffusers` library, write a function
`time_embedding_similarity(timesteps, embedding_dim, flip_sin_to_cos, downscale_freq_shift)`.

1. Embed all of `timesteps` in one call with `diffusers`' own sinusoidal time-embedding builder,
   `get_timestep_embedding` from `diffusers.models.embeddings`, into vectors of width
   `embedding_dim`, passing `flip_sin_to_cos` and `downscale_freq_shift` through to it unchanged and leaving every
   other argument at its default.
2. For each **consecutive pair** of timesteps, in the order given, compute the cosine similarity
   between their two embedding vectors.
3. Return those similarities as a list of floats, in pair order, each rounded to 4 decimal places.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: timesteps (list_int)
    Line 1: N (list length)
    Next N lines: one integer each
  Argument 2: embedding_dim (integer)
    Single line: one integer
  Argument 3: flip_sin_to_cos (boolean)
    Single line: true or false
  Argument 4: downscale_freq_shift (float)
    Single line: one float
```

**Output Format:**

For each test case, print the result of `time_embedding_similarity()` on a new line.

**Sample Input:**

```
timesteps = [950, 900, 800, 650, 0]
embedding_dim = 320
flip_sin_to_cos = True
downscale_freq_shift = 0.0
```

**Sample Output:**

```
[0.5124, 0.4433, 0.4172, 0.2207]
```

**Note:** `get_timestep_embedding` takes the timesteps as a torch tensor, and returns one row per
timestep. Return plain Python floats, not torch scalars.

---

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: time_embedding_similarity
RETURN TYPE: List of Floats

ARGUMENTS:
1. timesteps            → List of Ints
2. embedding_dim        → Integer
3. flip_sin_to_cos      → Boolean
4. downscale_freq_shift → Float
```

---

### 3. Reference Solution Code

```python
import torch
from diffusers.models.embeddings import get_timestep_embedding


def time_embedding_similarity(timesteps: list, embedding_dim: int,
                              flip_sin_to_cos: bool, downscale_freq_shift: float) -> list:
    embeddings = get_timestep_embedding(
        torch.tensor([int(t) for t in timesteps]),
        embedding_dim,
        flip_sin_to_cos=flip_sin_to_cos,
        downscale_freq_shift=downscale_freq_shift,
    )
    unit = embeddings / embeddings.norm(dim=-1, keepdim=True)
    similarities = (unit[:-1] * unit[1:]).sum(dim=-1)
    return [round(float(s), 4) for s in similarities]
```

---

### 4. Test Cases

**in1.txt**

```
1
5
950
900
800
650
0
320
true
0.0
```

**out1.txt**

```
[0.5124, 0.4433, 0.4172, 0.2207]
```

**in2.txt**

```
1
4
0
100
500
999
32
false
1.0
```

**out2.txt**

```
[0.3052, 0.1931, 0.3156]
```


In [1]:
import torch
from diffusers.models.embeddings import get_timestep_embedding


def time_embedding_similarity(timesteps: list, embedding_dim: int,
                              flip_sin_to_cos: bool, downscale_freq_shift: float) -> list:
    embeddings = get_timestep_embedding(
        torch.tensor([int(t) for t in timesteps]),
        embedding_dim,
        flip_sin_to_cos=flip_sin_to_cos,
        downscale_freq_shift=downscale_freq_shift,
    )
    unit = embeddings / embeddings.norm(dim=-1, keepdim=True)
    similarities = (unit[:-1] * unit[1:]).sum(dim=-1)
    return [round(float(s), 4) for s in similarities]


In [2]:
timesteps = [950, 900, 800, 650, 0]
embedding_dim = 320
flip_sin_to_cos = True
downscale_freq_shift = 0.0

time_embedding_similarity(timesteps, embedding_dim, flip_sin_to_cos, downscale_freq_shift)


[0.5124, 0.4433, 0.4172, 0.2207]

In [3]:
timesteps = [0, 100, 500, 999]
embedding_dim = 32
flip_sin_to_cos = False
downscale_freq_shift = 1.0

time_embedding_similarity(timesteps, embedding_dim, flip_sin_to_cos, downscale_freq_shift)


[0.3052, 0.1931, 0.3156]

### [Class 18] Question 16. What the Noise Schedule Decides

**Context:** The forward process is a fixed recipe, not something the model learns, and the recipe is
the noise schedule. Two schedules that both end at pure noise can take very different routes to get
there, and the route decides how long the image stays recognisable — which is the whole reason more
than one schedule exists. The quantity that settles the argument is the coefficient the forward
process applies to the clean image at step `t`: while it is near 1 the image is barely touched, and
once it approaches 0 there is nothing left to recover.

**Problem Statement:**

Using the `diffusers` library, write a function
`schedule_survival(schedule_names, thresholds, num_train_timesteps, beta_start, beta_end)`.

1. For each name in `schedule_names`, build a `diffusers.DDPMScheduler` with the given
   `num_train_timesteps`, `beta_start`, `beta_end`, and that name as its beta schedule. Leave every
   other scheduler setting at its default.
2. The scheduler exposes the cumulative product of the schedule as `alphas_cumprod`. The fraction of
   the clean image still present at step `t` is the **square root** of that cumulative product at `t`
   — the coefficient the forward process multiplies the clean image by. Timestep `t` is index `t` of
   `alphas_cumprod`.
3. For each value in `thresholds`, in the order given, find the **smallest** timestep at which that
   surviving fraction has dropped **below** the threshold. Every threshold given is crossed before the
   schedule ends.
4. Return a dictionary keyed by schedule name, whose value is the list of those timesteps, one per
   threshold, in the same order as `thresholds`.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: schedule_names (list_string)
    Line 1: N (list length)
    Next N lines: one string each
  Argument 2: thresholds (list_float)
    Line 1: M (list length)
    Next M lines: one float each
  Argument 3: num_train_timesteps (integer)
    Single line: one integer
  Argument 4: beta_start (float)
    Single line: one float
  Argument 5: beta_end (float)
    Single line: one float
```

**Output Format:**

For each test case, print the result of `schedule_survival()` on a new line.

**Sample Input:**

```
schedule_names = ["linear", "squaredcos_cap_v2"]
thresholds = [0.5, 0.1]
num_train_timesteps = 1000
beta_start = 0.0001
beta_end = 0.02
```

**Sample Output:**

```
{'linear': [368, 673], 'squaredcos_cap_v2': [664, 935]}
```

**Note:** The input handling is already in place — you only need to implement the function. Timesteps
are plain Python integers. Read the sample as a sanity check on what the two schedules are doing: one
of them is down to half the original image by step 368 while the other still holds half of it at 664,
which is the difference the choice of schedule buys.

---

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: schedule_survival
RETURN TYPE: Dictionary

ARGUMENTS:
1. schedule_names      → List of Strings
2. thresholds          → List of Floats
3. num_train_timesteps → Integer
4. beta_start          → Float
5. beta_end            → Float
```

---

### 3. Reference Solution Code

```python
from diffusers import DDPMScheduler


def schedule_survival(schedule_names: list, thresholds: list, num_train_timesteps: int,
                      beta_start: float, beta_end: float) -> dict:
    report = {}
    for name in schedule_names:
        scheduler = DDPMScheduler(
            num_train_timesteps=num_train_timesteps,
            beta_start=beta_start,
            beta_end=beta_end,
            beta_schedule=name,
        )
        surviving = scheduler.alphas_cumprod.sqrt()
        report[name] = [int((surviving < threshold).nonzero()[0]) for threshold in thresholds]
    return report
```

---

### 4. Test Cases

**in1.txt**

```
1
2
linear
squaredcos_cap_v2
2
0.5
0.1
1000
0.0001
0.02
```

**out1.txt**

```
{'linear': [368, 673], 'squaredcos_cap_v2': [664, 935]}
```

**in2.txt**

```
1
2
squaredcos_cap_v2
linear
2
0.9
0.5
500
0.001
0.05
```

**out2.txt**

```
{'squaredcos_cap_v2': [140, 332], 'linear': [56, 158]}
```


In [1]:
from diffusers import DDPMScheduler


def schedule_survival(schedule_names: list, thresholds: list, num_train_timesteps: int,
                      beta_start: float, beta_end: float) -> dict:
    report = {}
    for name in schedule_names:
        scheduler = DDPMScheduler(
            num_train_timesteps=num_train_timesteps,
            beta_start=beta_start,
            beta_end=beta_end,
            beta_schedule=name,
        )
        surviving = scheduler.alphas_cumprod.sqrt()
        report[name] = [int((surviving < threshold).nonzero()[0]) for threshold in thresholds]
    return report


In [2]:
schedule_names = ['linear', 'squaredcos_cap_v2']
thresholds = [0.5, 0.1]
num_train_timesteps = 1000
beta_start = 0.0001
beta_end = 0.02

schedule_survival(schedule_names, thresholds, num_train_timesteps, beta_start, beta_end)


{'linear': [368, 673], 'squaredcos_cap_v2': [664, 935]}

In [3]:
schedule_names = ['squaredcos_cap_v2', 'linear']
thresholds = [0.9, 0.5]
num_train_timesteps = 500
beta_start = 0.001
beta_end = 0.05

schedule_survival(schedule_names, thresholds, num_train_timesteps, beta_start, beta_end)


{'squaredcos_cap_v2': [140, 332], 'linear': [56, 158]}

### [Class 18] Question 17. Where a U-Net's Parameters Actually Sit

**Context:** The network doing the denoising is a U-Net, and its shape is the reason it can predict
noise at all: an encoder that walks the image down to a low resolution where it can reason about the
whole scene, a bottleneck at the bottom, and a decoder that walks back up while recovering detail.
The decoder does not work from the bottleneck alone — skip connections hand it the encoder's feature
maps at matching resolutions, and that has a consequence you can read straight off a parameter count.
Two other details only the built model will tell you: how many levels actually halve the resolution,
and how much of the network exists purely to encode the timestep.

**Problem Statement:**

Using the `diffusers` library, write a function
`unet_stage_profile(sample_size, in_channels, block_out_channels, layers_per_block)`.

1. Build a `diffusers.UNet2DModel` for `sample_size` × `sample_size` inputs with `in_channels`
   channels in and the same number out. `block_out_channels` gives the channel width at each level,
   so its length is the number of levels; use a plain `"DownBlock2D"` at every level of the encoder
   and a plain `"UpBlock2D"` at every level of the decoder, and pass `layers_per_block` through — the
   block lists go in as `down_block_types` and `up_block_types`, and the output width as
   `out_channels`. Leave every other setting at its default.
2. Count the parameters in each of the model's down blocks, in each of its up blocks, in its mid
   block, in its timestep-embedding module and in its input convolution — `down_blocks`, `up_blocks`,
   `mid_block`, `time_embedding` and `conv_in`.
3. Count the down blocks whose `downsamplers` is not `None`. The resolution at the bottleneck is
   `sample_size` halved once per such level.
4. Return a dictionary with exactly these seven keys:
   - `"down_params"` — the per-down-block parameter counts, encoder order
   - `"up_params"` — the per-up-block parameter counts, in `up_blocks` order (deepest level first)
   - `"mid_params"` — the mid block's parameter count
   - `"time_embedding_params"` — the timestep-embedding module's parameter count
   - `"conv_in_params"` — the input convolution's parameter count
   - `"downsampling_stages"` — how many down blocks have a `downsamplers` module
   - `"bottleneck_resolution"` — the spatial resolution at the bottleneck

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: sample_size (integer)
    Single line: one integer
  Argument 2: in_channels (integer)
    Single line: one integer
  Argument 3: block_out_channels (list_int)
    Line 1: N (list length)
    Next N lines: one integer each
  Argument 4: layers_per_block (integer)
    Single line: one integer
```

**Output Format:**

For each test case, print the result of `unet_stage_profile()` on a new line.

**Sample Input:**

```
sample_size = 64
in_channels = 4
block_out_channels = [32, 64, 128]
layers_per_block = 1
```

**Sample Output:**

```
{'down_params': [32000, 102912, 246656], 'up_params': [1051136, 312704, 78528], 'mid_params': 690688, 'time_embedding_params': 20736, 'conv_in_params': 1184, 'downsampling_stages': 2, 'bottleneck_resolution': 16}
```

**Note:** The input handling is already in place — you only need to implement the function. Every
value is a plain Python integer. No weights are downloaded: the model is constructed from the
configuration you pass, and parameter counts do not depend on how those parameters were initialised.

---

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: unet_stage_profile
RETURN TYPE: Dictionary

ARGUMENTS:
1. sample_size        → Integer
2. in_channels        → Integer
3. block_out_channels → List of Ints
4. layers_per_block   → Integer
```

---

### 3. Reference Solution Code

```python
from diffusers import UNet2DModel


def unet_stage_profile(sample_size: int, in_channels: int, block_out_channels: list,
                       layers_per_block: int) -> dict:
    levels = len(block_out_channels)
    unet = UNet2DModel(
        sample_size=sample_size,
        in_channels=in_channels,
        out_channels=in_channels,
        block_out_channels=tuple(block_out_channels),
        layers_per_block=layers_per_block,
        down_block_types=("DownBlock2D",) * levels,
        up_block_types=("UpBlock2D",) * levels,
    )

    count = lambda module: sum(p.numel() for p in module.parameters())
    stages = sum(1 for block in unet.down_blocks if block.downsamplers is not None)

    return {"down_params": [count(b) for b in unet.down_blocks],
            "up_params": [count(b) for b in unet.up_blocks],
            "mid_params": count(unet.mid_block),
            "time_embedding_params": count(unet.time_embedding),
            "conv_in_params": count(unet.conv_in),
            "downsampling_stages": stages,
            "bottleneck_resolution": sample_size // 2 ** stages}
```

---

### 4. Test Cases

**in1.txt**

```
1
64
4
3
32
64
128
1
```

**out1.txt**

```
{'down_params': [32000, 102912, 246656], 'up_params': [1051136, 312704, 78528], 'mid_params': 690688, 'time_embedding_params': 20736, 'conv_in_params': 1184, 'downsampling_stages': 2, 'bottleneck_resolution': 16}
```

**in2.txt**

```
1
64
4
2
64
128
2
```

**out2.txt**

```
{'down_params': [218048, 591616], 'up_params': [1593088, 448512], 'mid_params': 723456, 'time_embedding_params': 82432, 'conv_in_params': 2368, 'downsampling_stages': 1, 'bottleneck_resolution': 32}
```

**in3.txt**

```
1
128
3
4
32
64
96
128
1
```

**out3.txt**

```
{'down_params': [32000, 102912, 240416, 287680], 'up_params': [1092160, 644064, 292160, 78528], 'mid_params': 690688, 'time_embedding_params': 20736, 'conv_in_params': 896, 'downsampling_stages': 3, 'bottleneck_resolution': 16}
```


In [1]:
from diffusers import UNet2DModel


def unet_stage_profile(sample_size: int, in_channels: int, block_out_channels: list,
                       layers_per_block: int) -> dict:
    levels = len(block_out_channels)
    unet = UNet2DModel(
        sample_size=sample_size,
        in_channels=in_channels,
        out_channels=in_channels,
        block_out_channels=tuple(block_out_channels),
        layers_per_block=layers_per_block,
        down_block_types=("DownBlock2D",) * levels,
        up_block_types=("UpBlock2D",) * levels,
    )

    count = lambda module: sum(p.numel() for p in module.parameters())
    stages = sum(1 for block in unet.down_blocks if block.downsamplers is not None)

    return {"down_params": [count(b) for b in unet.down_blocks],
            "up_params": [count(b) for b in unet.up_blocks],
            "mid_params": count(unet.mid_block),
            "time_embedding_params": count(unet.time_embedding),
            "conv_in_params": count(unet.conv_in),
            "downsampling_stages": stages,
            "bottleneck_resolution": sample_size // 2 ** stages}


In [2]:
sample_size = 64
in_channels = 4
block_out_channels = [32, 64, 128]
layers_per_block = 1

unet_stage_profile(sample_size, in_channels, block_out_channels, layers_per_block)


{'down_params': [32000, 102912, 246656],
 'up_params': [1051136, 312704, 78528],
 'mid_params': 690688,
 'time_embedding_params': 20736,
 'conv_in_params': 1184,
 'downsampling_stages': 2,
 'bottleneck_resolution': 16}

In [3]:
sample_size = 64
in_channels = 4
block_out_channels = [64, 128]
layers_per_block = 2

unet_stage_profile(sample_size, in_channels, block_out_channels, layers_per_block)


{'down_params': [218048, 591616],
 'up_params': [1593088, 448512],
 'mid_params': 723456,
 'time_embedding_params': 82432,
 'conv_in_params': 2368,
 'downsampling_stages': 1,
 'bottleneck_resolution': 32}

In [4]:
sample_size = 128
in_channels = 3
block_out_channels = [32, 64, 96, 128]
layers_per_block = 1

unet_stage_profile(sample_size, in_channels, block_out_channels, layers_per_block)


{'down_params': [32000, 102912, 240416, 287680],
 'up_params': [1092160, 644064, 292160, 78528],
 'mid_params': 690688,
 'time_embedding_params': 20736,
 'conv_in_params': 896,
 'downsampling_stages': 3,
 'bottleneck_resolution': 16}

### [Class 19] Question 18. How Much of the Input Survives at Each `strength`

**Context:** Image-to-image does not blend the input with the output. It noises the input part of the
way up the ladder and starts denoising from there, so `strength` is a *noise level*, not an opacity.
That is why the parameter behaves so unevenly: the difference between 0.2 and 0.4 is a light retouch
against a real edit, while 0.9 and 1.0 both amount to starting over. Anyone tuning the dial is really
asking one question — how much of my image is still in there when the loop starts — and the schedule
answers it exactly.

**Problem Statement:**

Using the `diffusers` library, write a function
`strength_survival(strengths, num_inference_steps, num_train_timesteps)`.

1. Build a `diffusers.DDIMScheduler` with the given `num_train_timesteps` and every other scheduler
   setting left at the library default — `timestep_spacing` included, which means `"leading"`; do not
   carry over an override from anywhere else. Call `set_timesteps(num_inference_steps)` on it and read
   the schedule off `scheduler.timesteps`, which runs from noisiest to cleanest.
2. For each value in `strengths`, work out how much of that schedule actually runs: the number of steps
   is `num_inference_steps * strength` **rounded down** to a whole number — every `strength` is no larger than 1 and
   large enough that at least one step runs, so that count is between 1 and `num_inference_steps` — and the steps that run are
   the **last** ones — the high-noise front of the schedule is skipped, which is what keeps the result
   close to the input.
3. The loop therefore starts at the first timestep that actually runs. The scheduler exposes the
   cumulative product of its schedule as `alphas_cumprod`; the fraction of the input still present at
   a timestep is the **square root** of that cumulative product at that timestep.
4. Return a dictionary mapping each value in `strengths` to a two-element list: the number of steps
   that run, then the fraction of the input still present at the timestep the loop starts from,
   rounded to 4 decimal places.

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: strengths (list_float)
    Line 1: N (list length)
    Next N lines: one float each
  Argument 2: num_inference_steps (integer)
    Single line: one integer
  Argument 3: num_train_timesteps (integer)
    Single line: one integer
```

**Output Format:**

For each test case, print the result of `strength_survival()` on a new line.

**Sample Input:**

```
strengths = [0.2, 0.7, 1.0]
num_inference_steps = 30
num_train_timesteps = 1000
```

**Sample Output:**

```
{0.2: [6, 0.8651], 0.7: [21, 0.1091], 1.0: [30, 0.0096]}
```

**Note:** The input handling is already in place — you only need to implement the function. Step counts
are plain Python integers and fractions plain Python floats. Read the sample as a sanity check on why
the dial feels so uneven: at 0.2 most of the input is still there, at 0.7 only about a tenth of it is,
and at 1.0 there is essentially nothing left to preserve.

---

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: strength_survival
RETURN TYPE: Dictionary

ARGUMENTS:
1. strengths           → List of Floats
2. num_inference_steps → Integer
3. num_train_timesteps → Integer
```

---

### 3. Reference Solution Code

```python
from diffusers import DDIMScheduler


def strength_survival(strengths: list, num_inference_steps: int,
                      num_train_timesteps: int) -> dict:
    scheduler = DDIMScheduler(num_train_timesteps=num_train_timesteps)
    scheduler.set_timesteps(num_inference_steps)
    surviving = scheduler.alphas_cumprod.sqrt()

    report = {}
    for strength in strengths:
        steps_run = int(num_inference_steps * strength)
        first = num_inference_steps - steps_run
        start_timestep = int(scheduler.timesteps[first])
        report[strength] = [steps_run, round(float(surviving[start_timestep]), 4)]
    return report
```

---

### 4. Test Cases

**in1.txt**

```
1
3
0.2
0.7
1.0
30
1000
```

**out1.txt**

```
{0.2: [6, 0.8651], 0.7: [21, 0.1091], 1.0: [30, 0.0096]}
```

**in2.txt**

```
1
2
0.4
0.95
50
1000
```

**out2.txt**

```
{0.4: [20, 0.4762], 0.95: [47, 0.0137]}
```


In [1]:
from diffusers import DDIMScheduler


def strength_survival(strengths: list, num_inference_steps: int,
                      num_train_timesteps: int) -> dict:
    scheduler = DDIMScheduler(num_train_timesteps=num_train_timesteps)
    scheduler.set_timesteps(num_inference_steps)
    surviving = scheduler.alphas_cumprod.sqrt()

    report = {}
    for strength in strengths:
        steps_run = int(num_inference_steps * strength)
        first = num_inference_steps - steps_run
        start_timestep = int(scheduler.timesteps[first])
        report[strength] = [steps_run, round(float(surviving[start_timestep]), 4)]
    return report


In [2]:
strengths = [0.2, 0.7, 1.0]
num_inference_steps = 30
num_train_timesteps = 1000

strength_survival(strengths, num_inference_steps, num_train_timesteps)


{0.2: [6, 0.8651], 0.7: [21, 0.1091], 1.0: [30, 0.0096]}

In [3]:
strengths = [0.4, 0.95]
num_inference_steps = 50
num_train_timesteps = 1000

strength_survival(strengths, num_inference_steps, num_train_timesteps)


{0.4: [20, 0.4762], 0.95: [47, 0.0137]}

### [Class 19] Question 19. Proving ControlNet Starts by Doing Nothing

**Context:** ControlNet adds spatial control to a checkpoint without damaging it, and the claim that
it leaves the base model's behaviour intact is not a matter of trust — it is built into how the
adapter is initialised. ControlNet is a trainable copy of the U-Net's encoder that feeds its features
back into the frozen base model through a set of convolutions deliberately initialised to **zero**,
so that before any training happens the injected signal is exactly nothing and the base model behaves
as if the adapter were not attached. The conditioning image's own encoder ends the same way. That is a
property you can verify on a freshly built adapter in a few lines, and it is worth verifying, because
it is the entire safety argument for attaching one.

**Problem Statement:**

Using the `diffusers` library, write a function `controlnet_injection_profile(block_out_channels,
layers_per_block, cross_attention_dim, conditioning_embedding_out_channels)`.

1. Build a `diffusers.ControlNetModel` with 4 input channels, the given `block_out_channels` (whose
   length is the number of levels), `layers_per_block` and `cross_attention_dim`, a plain
   `"DownBlock2D"` at every level, and the given `conditioning_embedding_out_channels`. Leave every
   other setting at its default.
2. The convolutions that inject features back into the base model are the model's
   `controlnet_down_blocks`, plus its single `controlnet_mid_block`. Count them, and check whether
   **every** one of them has both weights and bias exactly zero.
3. Check whether the weights of the conditioning-image encoder's output convolution,
   `controlnet_cond_embedding.conv_out`, are zero too.
4. For contrast, check whether the weights of an ordinary convolution inside the adapter's own
   backbone — `down_blocks[0].resnets[0].conv1` — are zero.
5. Count the parameters in the adapter's own mid block, `mid_block`.
6. Return a dictionary with exactly these six keys:
   - `"n_injection_convs"` — how many injection convolutions there are, mid block included
   - `"injection_convs_all_zero"` — whether all of them are entirely zero
   - `"cond_embedding_output_zero"` — whether the conditioning encoder's output convolution is zero
   - `"backbone_conv_zero"` — whether the ordinary backbone convolution is zero
   - `"mid_block_params"` — the adapter's own mid block's parameter count
   - `"total_params"` — the adapter's total parameter count

**Input Format:**

```
Line 1: T (number of test cases)

For each test case:
  Argument 1: block_out_channels (list_int)
    Line 1: N (list length)
    Next N lines: one integer each
  Argument 2: layers_per_block (integer)
    Single line: one integer
  Argument 3: cross_attention_dim (integer)
    Single line: one integer
  Argument 4: conditioning_embedding_out_channels (list_int)
    Line 1: M (list length)
    Next M lines: one integer each
```

**Output Format:**

For each test case, print the result of `controlnet_injection_profile()` on a new line.

**Sample Input:**

```
block_out_channels = [32, 64]
layers_per_block = 1
cross_attention_dim = 32
conditioning_embedding_out_channels = [8, 16]
```

**Sample Output:**

```
{'n_injection_convs': 5, 'injection_convs_all_zero': True, 'cond_embedding_output_zero': True, 'backbone_conv_zero': False, 'mid_block_params': 252096, 'total_params': 390104}
```

**Note:** The input handling is already in place — you only need to implement the function. Return
plain Python integers and booleans. No weights are downloaded: the adapter is constructed from the
configuration you pass. Read the sample as a sanity check on the claim being tested: the contrast
between the injection convolutions and the ordinary backbone convolution is the answer to "why is
attaching a ControlNet safe?".

---

---

### 2. ML Function Descriptor Config

```
FUNCTION NAME: controlnet_injection_profile
RETURN TYPE: Dictionary

ARGUMENTS:
1. block_out_channels                  → List of Ints
2. layers_per_block                    → Integer
3. cross_attention_dim                 → Integer
4. conditioning_embedding_out_channels → List of Ints
```

---

### 3. Reference Solution Code

```python
from diffusers import ControlNetModel


def controlnet_injection_profile(block_out_channels: list, layers_per_block: int,
                                 cross_attention_dim: int,
                                 conditioning_embedding_out_channels: list) -> dict:
    levels = len(block_out_channels)
    controlnet = ControlNetModel(
        in_channels=4,
        block_out_channels=tuple(block_out_channels),
        layers_per_block=layers_per_block,
        down_block_types=("DownBlock2D",) * levels,
        cross_attention_dim=cross_attention_dim,
        conditioning_embedding_out_channels=tuple(conditioning_embedding_out_channels),
    )

    count = lambda module: sum(p.numel() for p in module.parameters())
    injection = list(controlnet.controlnet_down_blocks) + [controlnet.controlnet_mid_block]
    is_zero = lambda tensor: float(tensor.detach().abs().sum()) == 0.0
    backbone = controlnet.down_blocks[0].resnets[0].conv1.weight

    return {"n_injection_convs": len(injection),
            "injection_convs_all_zero": all(is_zero(c.weight) and is_zero(c.bias) for c in injection),
            "cond_embedding_output_zero": is_zero(controlnet.controlnet_cond_embedding.conv_out.weight),
            "backbone_conv_zero": is_zero(backbone),
            "mid_block_params": count(controlnet.mid_block),
            "total_params": count(controlnet)}
```

---

### 4. Test Cases

**in1.txt**

```
1
2
32
64
1
32
2
8
16
```

**out1.txt**

```
{'n_injection_convs': 5, 'injection_convs_all_zero': True, 'cond_embedding_output_zero': True, 'backbone_conv_zero': False, 'mid_block_params': 252096, 'total_params': 390104}
```

**in2.txt**

```
1
3
32
64
128
2
64
2
16
32
```

**out2.txt**

```
{'n_injection_convs': 10, 'injection_convs_all_zero': True, 'cond_embedding_output_zero': True, 'backbone_conv_zero': False, 'mid_block_params': 971136, 'total_params': 1874832}
```


In [1]:
from diffusers import ControlNetModel


def controlnet_injection_profile(block_out_channels: list, layers_per_block: int,
                                 cross_attention_dim: int,
                                 conditioning_embedding_out_channels: list) -> dict:
    levels = len(block_out_channels)
    controlnet = ControlNetModel(
        in_channels=4,
        block_out_channels=tuple(block_out_channels),
        layers_per_block=layers_per_block,
        down_block_types=("DownBlock2D",) * levels,
        cross_attention_dim=cross_attention_dim,
        conditioning_embedding_out_channels=tuple(conditioning_embedding_out_channels),
    )

    count = lambda module: sum(p.numel() for p in module.parameters())
    injection = list(controlnet.controlnet_down_blocks) + [controlnet.controlnet_mid_block]
    is_zero = lambda tensor: float(tensor.detach().abs().sum()) == 0.0
    backbone = controlnet.down_blocks[0].resnets[0].conv1.weight

    return {"n_injection_convs": len(injection),
            "injection_convs_all_zero": all(is_zero(c.weight) and is_zero(c.bias) for c in injection),
            "cond_embedding_output_zero": is_zero(controlnet.controlnet_cond_embedding.conv_out.weight),
            "backbone_conv_zero": is_zero(backbone),
            "mid_block_params": count(controlnet.mid_block),
            "total_params": count(controlnet)}


In [2]:
block_out_channels = [32, 64]
layers_per_block = 1
cross_attention_dim = 32
conditioning_embedding_out_channels = [8, 16]

controlnet_injection_profile(block_out_channels, layers_per_block, cross_attention_dim, conditioning_embedding_out_channels)


{'n_injection_convs': 5,
 'injection_convs_all_zero': True,
 'cond_embedding_output_zero': True,
 'backbone_conv_zero': False,
 'mid_block_params': 252096,
 'total_params': 390104}

In [3]:
block_out_channels = [32, 64, 128]
layers_per_block = 2
cross_attention_dim = 64
conditioning_embedding_out_channels = [16, 32]

controlnet_injection_profile(block_out_channels, layers_per_block, cross_attention_dim, conditioning_embedding_out_channels)


{'n_injection_convs': 10,
 'injection_convs_all_zero': True,
 'cond_embedding_output_zero': True,
 'backbone_conv_zero': False,
 'mid_block_params': 971136,
 'total_params': 1874832}